*Nombre de Grupos*:

*Integrantes*:

# **Laboratorio: Texturas y LBP**

En el laboratorio anterior decidíamos a qué región pertenecía cada píxel mirando su intensidad (umbralizar) o qué tan parecido era a sus vecinos (cortes de grafos). Pero hay imágenes donde lo importante no es *qué tan claro* es un píxel sino **cómo se organiza** respecto a los que tiene alrededor: una pared de ladrillos, el pasto, la piel de una cara. A eso lo llamamos **textura**.

En este laboratorio vamos a ver **LBP** (*Local Binary Patterns*), una forma clásica de describirla: compara cada píxel con su vecindad y cuenta qué patrones aparecen. Vamos a terminar reconociendo caras.

## **Imports**

In [ ]:
# === IMPORTS ===
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, IntSlider
from skimage import util, io, transform
from skimage.feature import local_binary_pattern

## **Carga de Imágenes**

Vamos a usar dos conjuntos de imágenes:

- **Texturas**: ocho fotos de texturas (manchadas, rotas, fibrosas...) del dataset DTD.
- **FERET**: un subconjunto de la base de rostros FERET, con 200 personas y 7 fotos de cada una (de 80x80 píxeles).

Esta celda los descarga (funciona en Colab y localmente). Si las carpetas ya existen no vuelve a descargar nada.

In [ ]:
#@title Descargar las imagenes { display-mode: "form" }
import io as _io
import os
import urllib.request
import zipfile

REPOSITORIO_URL = "https://github.com/Elementos-Reconocimiento-Visual-UBA/PracticasERV_2026_4toBim"  #@param {type:"string"}
RAMA = "main"  #@param {type:"string"}
CARPETA_EN_REPOSITORIO = "Laboratorio_3_Texturas/imagenes"  #@param {type:"string"}
FERET_URL = "https://github.com/VILAlab/FERETsubset/raw/master/FERET_80x80.zip"  #@param {type:"string"}
CARPETA_IMAGENES = "imagenes"  # carpeta local donde quedan las texturas
CARPETA_FERET = "feret"  # carpeta local donde queda FERET: una carpeta por persona (FERET-001/01.tif)

def leer_zip(url_zip):
  with urllib.request.urlopen(url_zip) as respuesta:
    return zipfile.ZipFile(_io.BytesIO(respuesta.read()))

def descargar_carpeta_de_github(url_zip, carpeta_en_repositorio, carpeta_local):
  # El zip de GitHub tiene todo dentro de una carpeta raiz: <repo>-<rama>/<carpeta_en_repositorio>/archivo
  zip_repositorio = leer_zip(url_zip)

  os.makedirs(carpeta_local, exist_ok=True)
  for nombre in zip_repositorio.namelist():
    # Sacamos la carpeta raiz y nos quedamos con los archivos que estan directamente en la carpeta pedida
    ruta = nombre.split("/", 1)[1] if "/" in nombre else ""
    carpeta, _, archivo = ruta.rpartition("/")
    if carpeta == carpeta_en_repositorio.strip("/") and archivo != "":
      with open(os.path.join(carpeta_local, archivo), "wb") as salida:
        salida.write(zip_repositorio.read(nombre))

def descargar_feret(url_zip, carpeta_local):
  # El zip de FERET tiene una carpeta raiz FERET_80x80 con una carpeta por persona
  zip_feret = leer_zip(url_zip)

  for nombre in zip_feret.namelist():
    ruta = nombre.split("/", 1)[1] if "/" in nombre else ""
    if ruta.endswith(".tif"):
      destino = os.path.join(carpeta_local, ruta)
      os.makedirs(os.path.dirname(destino), exist_ok=True)
      with open(destino, "wb") as salida:
        salida.write(zip_feret.read(nombre))

if os.path.isdir(CARPETA_IMAGENES):
  print(f"La carpeta {CARPETA_IMAGENES} ya existe, no se descarga nada")
else:
  descargar_carpeta_de_github(f"{REPOSITORIO_URL.rstrip('/')}/archive/refs/heads/{RAMA}.zip", CARPETA_EN_REPOSITORIO, CARPETA_IMAGENES)
  print(f"Descargadas {len(os.listdir(CARPETA_IMAGENES))} imagenes en {CARPETA_IMAGENES}")

if os.path.isdir(CARPETA_FERET):
  print(f"La carpeta {CARPETA_FERET} ya existe, no se descarga nada")
else:
  descargar_feret(FERET_URL, CARPETA_FERET)
  print(f"Descargadas las fotos de {len(os.listdir(CARPETA_FERET))} personas en {CARPETA_FERET}")

## **TEXTURAS**

### **Idea**

Una **textura** es un patrón que se repite a lo largo de una región: las grietas de la tierra seca, las fibras de un papel, las manchas de una pared. Lo que la define no es el brillo de ningún píxel en particular sino **cómo se relacionan los píxeles entre sí**.

Esto hace que las herramientas que vimos hasta ahora no funcionen. Un histograma cuenta *cuántos* píxeles hay de cada nivel de gris, pero no dice nada sobre *dónde* están. Veamos con un experimento que sucederia si utilizaramos un histograma para identificar una textura: vamos a mezclar de lugar un porcentaje de los píxeles y mirar qué pasa con la imagen y con su histograma.

In [ ]:
TEXTURAS = {
  'manchado 1': 'blotchy_0009.jpg',  'manchado 2': 'blotchy_0017.jpg',
  'roto 1':     'cracked_0004.jpg',  'roto 2':     'cracked_0046.jpg',
  'fibroso 1':  'fibrous_0083.jpg',  'fibroso 2':  'fibrous_0095.jpg',
  'moteado':    'flecked_0050.jpg',  'cristalino': 'crystalline_0057.jpg',
}
LADO_TEXTURA = 192

def cargar_textura(nombre, lado=LADO_TEXTURA):
  # Nos quedamos con un recorte cuadrado del centro de la foto
  imagen = util.img_as_ubyte(io.imread(f"{CARPETA_IMAGENES}/{TEXTURAS[nombre]}", as_gray=True))
  fila_inicio = (imagen.shape[0] - lado) // 2
  columna_inicio = (imagen.shape[1] - lado) // 2
  return imagen[fila_inicio:fila_inicio + lado, columna_inicio:columna_inicio + lado]

def mezclar_pixeles(imagen, porcentaje, semilla=0):
  generador = np.random.default_rng(semilla)
  mezclada = imagen.ravel().copy()
  cantidad = int(mezclada.size * porcentaje / 100)

  # Los pixeles elegidos se intercambian entre ellos: el histograma no puede cambiar
  posiciones = generador.choice(mezclada.size, cantidad, replace=False)
  mezclada[posiciones] = mezclada[generador.permutation(posiciones)]
  return mezclada.reshape(imagen.shape)

def mostrar_mezcla(textura, porcentaje):
  imagen = cargar_textura(textura)
  mezclada = mezclar_pixeles(imagen, porcentaje)

  fig, ax = plt.subplots(1, 3, figsize=(15, 4))

  ax[0].imshow(imagen, cmap="gray", vmin=0, vmax=255)
  ax[0].set_title("Imagen original")
  ax[0].axis("off")

  ax[1].imshow(mezclada, cmap="gray", vmin=0, vmax=255)
  ax[1].set_title(f"{porcentaje}% de los pixeles mezclados")
  ax[1].axis("off")

  ax[2].fill_between(range(256), np.bincount(imagen.ravel(), minlength=256), alpha=0.5, label="original")
  ax[2].plot(np.bincount(mezclada.ravel(), minlength=256), "r--", label="mezclada")
  ax[2].set_title("Histogramas")
  ax[2].set_xlim(0, 255)
  ax[2].legend()

  plt.tight_layout()
  plt.show()

interact(
  mostrar_mezcla,
  textura=widgets.Dropdown(options=list(TEXTURAS), value='roto 1', description='textura'),
  porcentaje=IntSlider(min=0, max=100, step=5, value=0, description='% mezclado'),
);

Fijense que a medida que mezclamos la textura se va deshaciendo hasta quedar convertida en ruido, pero **el histograma no se mueve**: es exactamente el mismo con 0% que con 100%.

Necesitamos descriptores que miren la **relación entre píxeles cercanos**:

- **LBP** mira cada píxel contra sus vecinos inmediatos y se queda con el *patrón* de la relacion relativa con ellos.

## **LBP (Local Binary Patterns)**

### **Idea**

**LBP** le asigna a cada píxel un número que describe cómo se compara con sus vecinos. No importa *cuánto* más claro u oscuro sea cada vecino, solo si es **mayor o igual** ($1$) o **menor** ($0$) que el píxel central.

Tomemos el píxel central $89$ y su vecindad de $3 \times 3$:

$$\begin{pmatrix} 100 & 23 & 59 \\ 34 & \textbf{89} & 2 \\ 97 & 140 & 4\end{pmatrix} $$

Comparando cada vecino contra el central nos queda

$$\begin{pmatrix} 1 & 0 & 0 \\ 0 & \textbf{89} & 0 \\ 1 & 1 & 0\end{pmatrix} $$

Para convertir esos ocho bits en **un solo número** hay que recorrer los vecinos siempre en el mismo orden. Vamos a empezar por el vecino de la derecha y dar la vuelta **en sentido antihorario** (recuerden que en una imagen las filas crecen hacia abajo). El vecino número $p$ aporta $2^p$ si es mayor o igual que el central:

$$LBP = \sum_{p=0}^{7} s(g_p - g_c)\, 2^p \qquad s(x) = \begin{cases} 1 & \text{si } x \ge 0 \\ 0 & \text{si } x < 0 \end{cases}$$

donde $g_c$ es el píxel central y $g_p$ su vecino $p$. En el ejemplo los vecinos $p=3$, $p=5$ y $p=6$ valen $1$, así que el código es $2^3 + 2^5 + 2^6 = 8 + 32 + 64 = 104$.

Hay una propiedad muy linda: como solo importa **quién es mayor que quién**, si a toda la imagen le cambiamos la iluminación con una función creciente (por ejemplo, multiplicarla por $2$ y sumarle $10$) los códigos **no cambian**. Eso hace a LBP muy robusto a la iluminación, y lo vamos a aprovechar más adelante.

Exploremos cómo se calcula el código en una textura real. Muevan la ventana de $3 \times 3$ y miren cómo cambian los bits.

In [ ]:
# Orden en que se recorren los 8 vecinos: (desplazamiento de fila, desplazamiento de columna)
# Arranca en el vecino de la derecha y sigue en sentido antihorario
OFFSETS_VECINOS_3X3 = [(0, 1), (-1, 1), (-1, 0), (-1, -1), (0, -1), (1, -1), (1, 0), (1, 1)]

imagen_ejemplo_lbp = cargar_textura('roto 1')[60:84, 60:84]

def explorar_lbp_basico(fila, columna):
  centro = imagen_ejemplo_lbp[fila, columna]
  vecinos = np.array([imagen_ejemplo_lbp[fila + df, columna + dc] for df, dc in OFFSETS_VECINOS_3X3])
  bits = (vecinos >= centro).astype(int)
  codigo_lbp = int(bits @ 2 ** np.arange(8))

  fig, ax = plt.subplots(1, 3, figsize=(15, 5))

  # Recuadro de la ventana sobre el recorte
  ax[0].imshow(imagen_ejemplo_lbp, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
  ax[0].plot([columna - 1.5, columna + 1.5, columna + 1.5, columna - 1.5, columna - 1.5], [fila - 1.5, fila - 1.5, fila + 1.5, fila + 1.5, fila - 1.5], color="orange", linewidth=2)
  ax[0].set_title("Recorte de la textura")
  ax[0].axis("off")

  # Valores de la vecindad
  ventana = imagen_ejemplo_lbp[fila - 1:fila + 2, columna - 1:columna + 2]
  ax[1].imshow(ventana, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
  for f in range(3):
    for c in range(3):
      color_texto = "black" if ventana[f, c] > 127 else "white"
      ax[1].text(c, f, int(ventana[f, c]), ha="center", va="center", color=color_texto, fontsize=14)
  ax[1].set_title(f"Vecindad (centro = {int(centro)})")
  ax[1].axis("off")

  # Bit de cada vecino y lo que aporta al codigo
  ax[2].imshow(np.zeros((3, 3)), cmap="gray", vmin=0, vmax=1)
  ax[2].text(1, 1, "centro", ha="center", va="center", color="white", fontsize=12)
  for posicion, ((df, dc), bit) in enumerate(zip(OFFSETS_VECINOS_3X3, bits)):
    ax[2].text(1 + dc, 1 + df, f"p={posicion}\nbit {bit}\n2^{posicion}={2 ** posicion}" if bit else f"p={posicion}\nbit {bit}", ha="center", va="center", color="lime" if bit else "red", fontsize=11)
  ax[2].set_title(f"LBP = {codigo_lbp}")
  ax[2].axis("off")

  plt.tight_layout()
  plt.show()

lado = imagen_ejemplo_lbp.shape[0]
interact(
  explorar_lbp_basico,
  fila=IntSlider(min=1, max=lado - 2, step=1, value=12, description='fila'),
  columna=IntSlider(min=1, max=lado - 2, step=1, value=12, description='columna'),
);

### ***Ejercicio***: Implementar LBP básico

Completar las funciones que calculan el código LBP de un píxel y el **mapa LBP** de una imagen entera (una imagen donde cada píxel es su código).

- Ejercicio: implementar
    - lbp_pixel
    - lbp_imagen

Ayudas:

- Ya está dada la lista `OFFSETS_VECINOS_3X3` con el orden de los vecinos. El vecino número $p$ de la lista aporta $2^p$.
- Un vecino **igual** al central cuenta como $1$ (es *mayor o igual*).
- Como cada píxel necesita sus 8 vecinos, el mapa **pierde el borde** de la imagen: tiene $2$ filas y $2$ columnas menos.

In [ ]:
def lbp_pixel(imagen, fila, columna):
  # Devuelve el codigo LBP (de 0 a 255) del pixel (fila, columna)
  # COMPLETAR
  # Recorran los vecinos con enumerate(OFFSETS_VECINOS_3X3) y sumen 2 ** posicion cuando el vecino sea mayor o igual que el centro
  pass

In [ ]:
def lbp_imagen(imagen):
  # Devuelve el mapa LBP: un codigo por cada pixel que tiene los 8 vecinos
  alto, ancho = imagen.shape
  mapa_lbp = np.zeros((alto - 2, ancho - 2), dtype=np.uint8)

  # COMPLETAR
  # Recorran solo los pixeles que tienen los 8 vecinos

  return mapa_lbp

### Tests (LBP básico)

In [ ]:
# nombre: (vecindad de 3x3, codigo esperado). El centro siempre esta en la posicion [1, 1]
CODIGOS_LBP_ESPERADOS = {
  'vecinos_menores':      (np.array([[0, 0, 0], [0, 255, 0], [0, 0, 0]]),         0),
  'vecinos_mayores':      (np.array([[255, 255, 255], [255, 0, 255], [255, 255, 255]]), 255),
  'solo_derecha':         (np.array([[0, 0, 0], [0, 100, 200], [0, 0, 0]]),       1),    # el vecino p=0 es el de la derecha
  'solo_abajo':           (np.array([[0, 0, 0], [0, 100, 0], [0, 200, 0]]),       64),   # el vecino p=6 es el de abajo
  'arriba_y_abajo':       (np.array([[0, 200, 0], [0, 100, 0], [0, 200, 0]]),     68),   # 4 + 64
  'ejemplo_de_la_teoria': (np.array([[100, 23, 59], [34, 89, 2], [97, 140, 4]]),  104),
  'igual_al_centro':      (np.full((3, 3), 50),                                   255),  # los vecinos iguales al centro cuentan como 1
}

# Imagen de 3x4: solo tiene dos pixeles con los 8 vecinos, en una rampa los dos tienen el mismo patron
IMAGEN_RAMPA = np.array([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]], dtype=np.uint8)
MAPA_RAMPA_ESPERADO = np.array([[225, 225]])  # 1 + 32 + 64 + 128: solo son menores los vecinos de arriba y de la izquierda

def verificar_lbp_pixel(nombre):
  vecindad, esperado = CODIGOS_LBP_ESPERADOS[nombre]
  obtenido = lbp_pixel(vecindad, 1, 1)

  assert obtenido == esperado, f"[{nombre}] se esperaba {esperado} pero se obtuvo {obtenido}"
  print(f"OK [{nombre}] LBP = {obtenido}")

def verificar_lbp_imagen_rampa():
  obtenido = lbp_imagen(IMAGEN_RAMPA)

  assert obtenido.shape == MAPA_RAMPA_ESPERADO.shape, f"[rampa] se esperaba un mapa de forma {MAPA_RAMPA_ESPERADO.shape} pero se obtuvo {obtenido.shape}"
  assert np.array_equal(obtenido, MAPA_RAMPA_ESPERADO), f"[rampa] se esperaba {MAPA_RAMPA_ESPERADO.tolist()} pero se obtuvo {obtenido.tolist()}"
  print(f"OK [rampa] mapa = {obtenido.tolist()}")

def verificar_lbp_imagen_forma():
  imagen = np.random.default_rng(0).integers(0, 256, (7, 9)).astype(np.uint8)
  obtenido = lbp_imagen(imagen)

  assert obtenido.shape == (5, 7), f"[forma] se esperaba un mapa de forma (5, 7) pero se obtuvo {obtenido.shape}"
  assert obtenido[2, 3] == lbp_pixel(imagen, 3, 4), "[forma] el pixel (2, 3) del mapa tiene que ser el LBP del pixel (3, 4) de la imagen"
  print(f"OK [forma] una imagen de {imagen.shape} da un mapa de {obtenido.shape}")

def verificar_invarianza_a_la_iluminacion():
  imagen = np.random.default_rng(1).integers(0, 100, (12, 12)).astype(np.uint8)
  imagen_con_otra_iluminacion = 2 * imagen + 10

  assert np.array_equal(lbp_imagen(imagen), lbp_imagen(imagen_con_otra_iluminacion)), "el LBP de 2 * imagen + 10 tiene que ser igual al de la imagen"
  print("OK [iluminacion] 2 * imagen + 10 tiene el mismo mapa LBP")

### Verificacion LBP de un píxel

In [ ]:
for nombre in CODIGOS_LBP_ESPERADOS:
  verificar_lbp_pixel(nombre)

### Verificacion LBP de una imagen

In [ ]:
verificar_lbp_imagen_rampa()
verificar_lbp_imagen_forma()
verificar_invarianza_a_la_iluminacion()

#### **Ejemplo**

In [ ]:
textura = cargar_textura('roto 1')
mapa_lbp = lbp_imagen(textura)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))

ax[0].imshow(textura, cmap="gray")
ax[0].set_title("Textura original")
ax[0].axis("off")

ax[1].imshow(mapa_lbp, cmap="gray")
ax[1].set_title("Mapa LBP")
ax[1].axis("off")

ax[2].bar(range(256), np.bincount(mapa_lbp.ravel(), minlength=256), width=1)
ax[2].set_title("Histograma del mapa LBP")
ax[2].set_xlabel("codigo LBP")
ax[2].set_xlim(0, 255)

plt.tight_layout()
plt.show()

Ahora vamos a repetir con LBP el experimento del principio. Mezclamos de lugar un porcentaje de los píxeles de la textura, recalculamos su **mapa LBP** y comparamos el histograma de ese mapa contra el de la textura original.

In [ ]:
def mostrar_mezcla_en_lbp(porcentaje):
  # textura y mapa_lbp son los de la celda anterior
  mezclada = mezclar_pixeles(textura, porcentaje)
  mapa_lbp_mezclada = lbp_imagen(mezclada)
  histograma_original = np.bincount(mapa_lbp.ravel(), minlength=256) / mapa_lbp.size
  histograma_mezclada = np.bincount(mapa_lbp_mezclada.ravel(), minlength=256) / mapa_lbp_mezclada.size

  fig, ax = plt.subplots(1, 3, figsize=(15, 5))

  ax[0].imshow(mezclada, cmap="gray", vmin=0, vmax=255)
  ax[0].set_title(f"{porcentaje}% de los pixeles mezclados")
  ax[0].axis("off")

  ax[1].imshow(mapa_lbp_mezclada, cmap="gray")
  ax[1].set_title("Mapa LBP de la imagen mezclada")
  ax[1].axis("off")

  ax[2].fill_between(range(256), histograma_original, step="mid", alpha=0.5, label="original")
  ax[2].plot(histograma_mezclada, "r--", label="mezclada")
  ax[2].set_title(f"Histogramas del mapa LBP (diferencia {np.abs(histograma_original - histograma_mezclada).sum():.2f})")
  ax[2].set_xlabel("codigo LBP")
  ax[2].set_xlim(0, 255)
  ax[2].legend()

  plt.tight_layout()
  plt.show()

# El mapa LBP se calcula con loops: solo se recalcula al soltar el slider
interact(mostrar_mezcla_en_lbp, porcentaje=IntSlider(min=0, max=100, step=5, value=50, description='% mezclado', continuous_update=False));

Fijense que **ahora el histograma cambia**. Al mezclar los píxeles se destruyen las formas locales: los bordes y las esquinas ordenadas de la textura se convierten en ruido, y el ruido está lleno de puntos aislados que son máximos o mínimos locales. Por eso los códigos $0$ y $255$ pasan de sumar alrededor del $14\%$ de los píxeles a más del $20\%$.

El mapa de LBP nos da una idea de la distribucion de las estructuras geometricas de la imagen. Cada codigo LBP da un identificador unico a una estructura local en un vecindario, por ejemplo bordes y esquinas en diferentes direcciones, zonas homogeneas, zonas caoticas, etc.

Este variacion basica de LBP mira siempre un vecindario de $3 \times 3$. Si la textura tuviera detalles más grandes, con un vecindario tan chico no podrian captarse.

### **Vecindad circular (P, R)**

Supongamos que tomamos una textura y la **ampliamos** al triple: sigue siendo la misma textura, pero lo que antes eran detalles de $1$ píxel ahora son manchas de $3$ píxeles. Con un cuadrado de $3 \times 3$ casi todos los vecinos de un píxel caen dentro de la misma mancha y el código deja de describir la textura.

La solución es que la vecindad **no tenga un tamaño fijo**: en vez de los 8 píxeles pegados al central, tomamos $P$ puntos equiespaciados sobre un **círculo de radio $R$**. Con $R$ chico miramos detalles finos y con $R$ grande, estructuras generales. Seguimos el mismo orden de antes: arrancamos a la derecha y damos la vuelta en sentido antihorario. Los puntos están en

$$\Delta f_p = -R \cdot \sin\left(\frac{2\pi p}{P}\right) \qquad \Delta c_p = R \cdot \cos\left(\frac{2\pi p}{P}\right) \qquad p = 0, \dots, P-1$$

(el signo menos en la fila es porque las filas crecen hacia abajo).

Casi ningún punto del círculo cae justo sobre un píxel. Para saber cuánto vale la imagen **entre píxeles** usamos **interpolación bilineal**: el valor es un promedio de los 4 píxeles más cercanos, pesado por qué tan cerca está cada uno. Es lo mismo que hicimos en Hough para repartir un voto entre bins vecinos.

### ***Ejercicio***: Implementar LBP circular

Completar las funciones para calcular LBP con una vecindad de $P$ puntos y radio $R$.

- Ejercicio: implementar
    - coordenadas_vecinos
    - interpolacion_bilineal
    - lbp_circular

Ayudas:

- `coordenadas_vecinos` devuelve una matriz de $P \times 2$ con los $(\Delta f_p, \Delta c_p)$ de cada vecino. Siempre que comparen, usen una **tolerancia**: $|x - 0| < tol$.
- Para comparar valores utilizen `vecino - centro >= -tolerancia`.
- En `interpolacion_bilineal` las coordenadas pueden caer fuera de la imagen, está dada `indice_dentro_de_la_imagen` que las pega al borde.
- Igual que antes, cada píxel necesita todos sus vecinos, así que el mapa pierde un borde de $\lceil R \rceil$ píxeles.

In [ ]:
def indice_dentro_de_la_imagen(indice, limite):
  # Pega al borde los indices que se salen de la imagen
  return min(max(0, indice), limite - 1)

# Dos numeros con coma que deberian ser iguales pueden diferir en el orden de 1e-16: los comparamos con una tolerancia (esta es la que usan por defecto las funciones)
TOLERANCIA = 1e-9

In [ ]:
def coordenadas_vecinos(P, R, tolerancia=TOLERANCIA):
  # Devuelve una matriz de P x 2 con (desplazamiento de fila, desplazamiento de columna) de cada vecino
  # COMPLETAR
  pass

In [ ]:
def interpolacion_bilineal(imagen, fila, columna):
  # Valor de la imagen en (fila, columna), que pueden ser numeros con coma
  alto, ancho = imagen.shape

  # COMPLETAR
  pass

In [ ]:
def lbp_circular(imagen, P=8, R=1, tolerancia=TOLERANCIA):
  # Devuelve el mapa LBP con P vecinos de radio R. Pierde un borde de ceil(R) pixeles
  borde = int(np.ceil(R))
  alto, ancho = imagen.shape
  coordenadas = coordenadas_vecinos(P, R, tolerancia)
  mapa_lbp = np.zeros((alto - 2 * borde, ancho - 2 * borde), dtype=np.int64)

  # COMPLETAR
  # Para cada pixel interior: interpolen el valor de cada vecino y comparenlo con el central usando la tolerancia

  return mapa_lbp

### Tests (LBP circular)

In [ ]:
# nombre: (P, R, matriz esperada de P x 2)
COORDENADAS_ESPERADAS = {
  'cuatro_vecinos_radio_1': (4, 1, np.array([[0, 1], [-1, 0], [0, -1], [1, 0]])),
  'ocho_vecinos_radio_1':   (8, 1, np.array([[0, 1], [-np.sqrt(2) / 2, np.sqrt(2) / 2], [-1, 0], [-np.sqrt(2) / 2, -np.sqrt(2) / 2], [0, -1], [np.sqrt(2) / 2, -np.sqrt(2) / 2], [1, 0], [np.sqrt(2) / 2, np.sqrt(2) / 2]])),
  'cuatro_vecinos_radio_3': (4, 3, np.array([[0, 3], [-3, 0], [0, -3], [3, 0]])),
}

# nombre: (imagen, fila, columna, valor esperado). La imagen tiene 0 y 100 en la fila de arriba, 200 y 40 en la de abajo
IMAGEN_INTERPOLACION = np.array([[0, 100], [200, 40]])
INTERPOLACIONES_ESPERADAS = {
  'sobre_un_pixel':   (IMAGEN_INTERPOLACION, 1,    1,    40),
  'mitad_horizontal': (IMAGEN_INTERPOLACION, 0,    0.5,  50),
  'mitad_vertical':   (IMAGEN_INTERPOLACION, 0.5,  0,    100),
  'centro_de_cuatro': (IMAGEN_INTERPOLACION, 0.5,  0.5,  85),
  'pesos_distintos':  (IMAGEN_INTERPOLACION, 0.25, 0.75, 76.25),
  'fuera_por_arriba': (IMAGEN_INTERPOLACION, -3,   0.5,  50),     # se pega al borde de arriba
  'fuera_por_derecha': (IMAGEN_INTERPOLACION, 1,   7,    40),     # se pega al borde de la derecha
}

# En la cruz de P=4 y R=1 todos los vecinos caen sobre un pixel, asi que no hace falta interpolar
IMAGEN_CRUZ = np.array([[0, 5, 0], [1, 3, 9], [0, 2, 0]])  # derecha 9, arriba 5, izquierda 1, abajo 2, centro 3
CODIGO_CRUZ_ESPERADO = 3  # derecha (1) y arriba (2) son mayores o iguales que el centro

def verificar_coordenadas(nombre):
  P, R, esperadas = COORDENADAS_ESPERADAS[nombre]
  obtenidas = np.asarray(coordenadas_vecinos(P, R, TOLERANCIA))

  assert obtenidas.shape == (P, 2), f"[{nombre}] se esperaba una matriz de forma {(P, 2)} pero se obtuvo {obtenidas.shape}"
  assert np.allclose(obtenidas, esperadas, atol=TOLERANCIA), f"[{nombre}] se esperaba {esperadas.tolist()} pero se obtuvo {obtenidas.tolist()}"
  assert np.all(obtenidas[esperadas == 0] == 0), f"[{nombre}] las coordenadas que valen 0 tienen que valer exactamente 0 y no algo como 1e-16"
  print(f"OK [{nombre}] coordenadas de {P} vecinos con radio {R}")

def verificar_interpolacion(nombre):
  imagen, fila, columna, esperado = INTERPOLACIONES_ESPERADAS[nombre]
  obtenido = interpolacion_bilineal(imagen, fila, columna)

  assert np.isclose(obtenido, esperado), f"[{nombre}] se esperaba {esperado} pero se obtuvo {obtenido}"
  print(f"OK [{nombre}] valor en ({fila}, {columna}) = {obtenido}")

def verificar_lbp_circular_cruz():
  obtenido = lbp_circular(IMAGEN_CRUZ, 4, 1)

  assert obtenido.shape == (1, 1), f"[cruz] se esperaba un mapa de forma (1, 1) pero se obtuvo {obtenido.shape}"
  assert obtenido[0, 0] == CODIGO_CRUZ_ESPERADO, f"[cruz] se esperaba {CODIGO_CRUZ_ESPERADO} pero se obtuvo {obtenido[0, 0]}"
  print(f"OK [cruz] LBP = {obtenido[0, 0]}")

def verificar_lbp_circular_contra_skimage(P, R):
  # skimage usa el mismo orden de vecinos y la misma interpolacion, pero redondea las coordenadas a 5 decimales: puede diferir en algun pixel suelto
  # Comparamos solo el interior, donde estan todos los vecinos
  imagen = cargar_textura('fibroso 2', 32)
  borde = int(np.ceil(R))
  esperado = local_binary_pattern(imagen, P, R, method="default")[borde:-borde, borde:-borde]
  obtenido = lbp_circular(imagen, P, R)

  assert obtenido.shape == esperado.shape, f"[P={P}, R={R}] se esperaba un mapa de forma {esperado.shape} pero se obtuvo {obtenido.shape}"
  coincidencia = (obtenido == esperado).mean()
  assert coincidencia > 0.995, f"[P={P}, R={R}] solo coincide con skimage en el {coincidencia * 100:.1f}% de los pixeles"
  print(f"OK [P={P}, R={R}] coincide con skimage en el {coincidencia * 100:.1f}% de los pixeles")

def verificar_lbp_circular_plana(P, R):
  # En una imagen plana todos los vecinos valen lo mismo que el centro, y los iguales cuentan como 1: todos los bits tienen que ser 1
  imagen = np.full((12, 12), 50, dtype=np.uint8)
  obtenido = lbp_circular(imagen, P, R, TOLERANCIA)

  assert np.all(obtenido == 2 ** P - 1), f"[plana P={P}, R={R}] se esperaba {2 ** P - 1} en todos los pixeles pero se obtuvo {np.unique(obtenido).tolist()} (cuidado con comparar numeros con coma sin tolerancia)"
  print(f"OK [plana P={P}, R={R}] todos los codigos valen {2 ** P - 1}")

### Verificacion coordenadas

In [ ]:
for nombre in COORDENADAS_ESPERADAS:
  verificar_coordenadas(nombre)

### Verificacion interpolación bilineal

In [ ]:
for nombre in INTERPOLACIONES_ESPERADAS:
  verificar_interpolacion(nombre)

### Verificacion LBP circular

In [ ]:
verificar_lbp_circular_cruz()
for P, R in [(8, 1), (8, 2), (12, 3), (16, 2)]:
  verificar_lbp_circular_contra_skimage(P, R)
for P, R in [(8, 1), (8, 2), (16, 2)]:
  verificar_lbp_circular_plana(P, R)

#### Opcional: Vectorizado

Igual que en Hough, la versión con loops es lenta: interpola un punto a la vez. Intenten vectorizarla: para cada vecino $p$ calculamos de una sola vez la imagen completa de valores interpolados (una imagen del mismo tamaño que la original donde cada píxel vale lo que hay en su vecino $p$) y la comparamos contra la imagen original.

- Ejercicio Opcional: implementar
    - interpolar_imagen_desplazada
    - lbp_circular_vectorizado

Ayudas:

- Con `np.indices(imagen.shape)` consiguen las coordenadas de todos los píxeles de una vez, y con `np.clip` pegan al borde los índices que se salen.

In [ ]:
def interpolar_imagen_desplazada(imagen, desplazamiento_fila, desplazamiento_columna):
  # Devuelve una imagen del mismo tamaño que la original donde cada pixel (f, c) vale la interpolacion bilineal en (f + desplazamiento_fila, c + desplazamiento_columna)
  alto, ancho = imagen.shape

  # COMPLETAR
  pass

In [ ]:
def lbp_circular_vectorizado(imagen, P=8, R=1, tolerancia=TOLERANCIA):
  borde = int(np.ceil(R))
  codigos = np.zeros(imagen.shape, dtype=np.int64)

  # COMPLETAR

  return codigos[borde:-borde, borde:-borde]

### Verificacion Vectorizado (LBP circular)

In [ ]:
def verificar_lbp_circular_vectorizado(P, R):
  imagen = cargar_textura('fibroso 2', 32)

  inicio = time.perf_counter()
  esperado = lbp_circular(imagen, P, R)
  tiempo_loop = time.perf_counter() - inicio

  inicio = time.perf_counter()
  obtenido = lbp_circular_vectorizado(imagen, P, R)
  tiempo_vectorizado = time.perf_counter() - inicio

  assert obtenido.shape == esperado.shape, f"[P={P}, R={R}] se esperaba un mapa de forma {esperado.shape} pero se obtuvo {obtenido.shape}"
  assert np.array_equal(obtenido, esperado), f"[P={P}, R={R}] el mapa vectorizado no coincide con el de los loops en {(obtenido != esperado).sum()} pixeles"
  print(f"OK [P={P}, R={R}] loop: {tiempo_loop * 1000:.1f} ms, vectorizado: {tiempo_vectorizado * 1000:.1f} ms")

for P, R in [(8, 1), (8, 2), (12, 3)]:
  verificar_lbp_circular_vectorizado(P, R)

#### **Ejemplo: la escala**

Tomemos una textura, la **ampliamos** $z$ veces y comparamos el histograma LBP de la imagen original (con $R=1$) contra el de la imagen ampliada. Si la vecindad circular funciona, al aumentar $R$ hasta más o menos $z$ el histograma de la imagen ampliada deberia reducir la diferencia.

Como la versión con loops tarda mucho en imágenes grandes, ahora que verificamos que coincide con la de skimage vamos a usar `local_binary_pattern` de skimage.

In [ ]:
def histograma_lbp_rapido(imagen, P, R):
  mapa_lbp = local_binary_pattern(imagen, P, R, method="default").astype(int)
  return np.bincount(mapa_lbp.ravel(), minlength=2 ** P) / mapa_lbp.size

def explorar_escala(textura, zoom, R):
  original = cargar_textura(textura, 96)
  ampliada = util.img_as_ubyte(transform.resize(original, (96 * zoom, 96 * zoom), order=1, anti_aliasing=False))

  histograma_original = histograma_lbp_rapido(original, 8, 1)
  histograma_ampliada_r1 = histograma_lbp_rapido(ampliada, 8, 1)
  histograma_ampliada = histograma_lbp_rapido(ampliada, 8, R)
  diferencia_r1 = np.abs(histograma_original - histograma_ampliada_r1).sum()
  diferencia = np.abs(histograma_original - histograma_ampliada).sum()

  fig, ax = plt.subplots(1, 3, figsize=(15, 4))

  ax[0].imshow(original, cmap="gray", vmin=0, vmax=255)
  ax[0].set_title("Original (96x96)")
  ax[0].axis("off")

  ax[1].imshow(ampliada, cmap="gray", vmin=0, vmax=255)
  ax[1].set_title(f"Ampliada x{zoom} ({96 * zoom}x{96 * zoom})")
  ax[1].axis("off")

  ax[2].plot(histograma_original, color="black", label="original, R=1")
  ax[2].plot(histograma_ampliada_r1, color="red", alpha=0.6, label=f"ampliada, R=1 (diferencia {diferencia_r1:.2f})")
  ax[2].plot(histograma_ampliada, color="blue", alpha=0.6, label=f"ampliada, R={R} (diferencia {diferencia:.2f})")
  ax[2].set_title("Histogramas LBP (P=8)")
  ax[2].set_xlim(0, 255)
  ax[2].legend(fontsize=8)

  plt.tight_layout()
  plt.show()

interact(
  explorar_escala,
  textura=widgets.Dropdown(options=list(TEXTURAS), value='roto 1', description='textura'),
  zoom=IntSlider(min=1, max=4, step=1, value=3, description='zoom'),
  R=IntSlider(min=1, max=8, step=1, value=1, description='R'),
);

### **Patrones uniformes**

Con $P$ vecinos hay $2^P$ códigos posibles: $256$ para $P=8$ pero $65536$ para $P=16$. Como el histograma tiene un bin por código, mientras más vecinos usamos más grande y más **vacío** queda: casi todos los bines tienen cero o un puñado de píxeles y el histograma se vuelve ruidoso.

La observación que resuelve esto es empírica: la mayoría de los patrones que aparecen en texturas reales son muy ordenados, con todos los $1$ juntos y todos los $0$ juntos (un punto, un borde, una esquina). Para medir qué tan ordenado es un patrón contamos sus **transiciones**: las veces que el bit cambia de $0$ a $1$ o de $1$ a $0$ al recorrer el círculo. Como el círculo es cerrado, el último bit también se compara con el primero.

| Patrón | Transiciones | |
|---|---|---|
| `00011100` | 2 | uniforme: los unos están juntos |
| `10000011` | 2 | uniforme: los unos dan la vuelta por el borde |
| `00010110` | 4 | no uniforme |
| `01010101` | 8 | no uniforme |

Un patrón es **uniforme** si tiene **a lo sumo 2 transiciones**. La idea es darle un bin propio a cada patrón uniforme y **un único casillero** que junte a todos los no uniformes. Para $P=8$ eso baja de $256$ a $59$ casilleros ($58$ uniformes y $1$ para el resto).

Vamos a empezar contando transiciones y verificando cuántos patrones uniformes hay.

### ***Ejercicio***: Contar transiciones y patrones uniformes

- Ejercicio: implementar
    - contar_transiciones
    - contar_patrones_uniformes

Ayudas:

- Un patrón es una lista de $P$ bits. Ya está dada `bits_de_codigo`, que pasa de un código a su lista de bits (el bit $p$ es el del vecino $p$).
- El vecino que sigue al último es el primero.
- La cantidad de patrones uniformes tiene una fórmula cerrada: $P(P-1) + 2$. Fijense que sus resultados la cumplan.

In [ ]:
def bits_de_codigo(codigo_lbp, P):
  # El bit p del codigo es el del vecino p
  return [(codigo_lbp >> posicion) & 1 for posicion in range(P)]

In [ ]:
def contar_transiciones(patron_bits):
  # Cuenta cuantas veces cambia el bit al recorrer el circulo. El ultimo bit se compara con el primero
  # COMPLETAR
  pass

In [ ]:
def contar_patrones_uniformes(P):
  # Cuenta cuantos de los 2 ** P codigos posibles son uniformes (a lo sumo 2 transiciones)
  # COMPLETAR
  pass

### Tests (patrones uniformes)

In [ ]:
# nombre: (patron de 8 bits, transiciones esperadas)
TRANSICIONES_ESPERADAS = {
  'todos_ceros':          ([0, 0, 0, 0, 0, 0, 0, 0], 0),
  'todos_unos':           ([1, 1, 1, 1, 1, 1, 1, 1], 0),
  'un_solo_uno':          ([0, 0, 0, 1, 0, 0, 0, 0], 2),
  'unos_juntos':          ([0, 1, 1, 1, 0, 0, 0, 0], 2),
  'unos_dan_la_vuelta':   ([1, 1, 0, 0, 0, 0, 0, 1], 2),  # el ultimo bit se compara con el primero
  'cuatro_transiciones':  ([0, 1, 0, 1, 0, 0, 0, 0], 4),
  'alternados':           ([0, 1, 0, 1, 0, 1, 0, 1], 8),
  'ejemplo_de_la_teoria': ([0, 0, 0, 1, 0, 1, 1, 0], 4),  # el patron del 89 de la seccion de LBP basico
}

# P: cantidad de patrones uniformes esperada, que es P * (P - 1) + 2
UNIFORMES_ESPERADOS = {4: 14, 8: 58, 12: 134}

def verificar_transiciones(nombre):
  patron_bits, esperadas = TRANSICIONES_ESPERADAS[nombre]
  obtenidas = contar_transiciones(patron_bits)

  assert obtenidas == esperadas, f"[{nombre}] se esperaban {esperadas} transiciones pero se obtuvieron {obtenidas}"
  print(f"OK [{nombre}] {obtenidas} transiciones")

def verificar_patrones_uniformes(P):
  esperados = UNIFORMES_ESPERADOS[P]
  obtenidos = contar_patrones_uniformes(P)

  assert obtenidos == esperados, f"[P={P}] se esperaban {esperados} patrones uniformes pero se obtuvieron {obtenidos}"
  print(f"OK [P={P}] {obtenidos} patrones uniformes de {2 ** P}")

### Verificacion transiciones

In [ ]:
for nombre in TRANSICIONES_ESPERADAS:
  verificar_transiciones(nombre)

### Verificacion patrones uniformes

In [ ]:
for P in UNIFORMES_ESPERADOS:
  verificar_patrones_uniformes(P)

Ahora hay que darle a cada patrón uniforme su bin. Los ordenamos por **cantidad de unos** y, dentro de cada cantidad, por **rotación** (en qué posición arranca la fila de unos). El casillero $0$ es el de todos ceros, el $P(P-1)+1$ es el de todos unos y el último, $P(P-1)+2$, es el de los no uniformes. Para $P=8$ son las etiquetas $0$ a $58$ de skimage (`method="nri_uniform"`). Esta función ya está dada.

In [ ]:
def etiqueta_nri_uniforme(patron_bits):
  # Un casillero por patron uniforme y uno para todos los no uniformes
  P = len(patron_bits)
  if contar_transiciones(patron_bits) > 2: return P * (P - 1) + 2

  cantidad_de_unos = sum(patron_bits)
  if cantidad_de_unos == 0: return 0
  if cantidad_de_unos == P: return P * (P - 1) + 1

  # Rotacion: segun donde arranca la tira de unos (si arranca en 0 hay que mirar donde termina)
  primer_uno = patron_bits.index(1)
  primer_cero = patron_bits.index(0)
  rotacion = cantidad_de_unos - primer_cero if primer_uno == 0 else P - primer_uno
  return 1 + (cantidad_de_unos - 1) * P + rotacion

def tabla_de_etiquetas(P, funcion_etiqueta):
  # tabla[codigo_lbp] = etiqueta del patron, para los 2 ** P codigos posibles
  return np.array([funcion_etiqueta(bits_de_codigo(codigo_lbp, P)) for codigo_lbp in range(2 ** P)])

Verifiquemos que la tabla coincide con la de skimage: calculamos el mapa LBP común y lo re-etiquetamos con la tabla. Si todo está bien tiene que dar lo mismo que pedirle a skimage el método `nri_uniform`.

In [ ]:
def verificar_tabla_nri_contra_skimage(nombre_textura):
  imagen = cargar_textura(nombre_textura)
  mapa_comun = local_binary_pattern(imagen, 8, 1, method="default").astype(int)
  esperado = local_binary_pattern(imagen, 8, 1, method="nri_uniform")
  obtenido = tabla_de_etiquetas(8, etiqueta_nri_uniforme)[mapa_comun]

  assert np.array_equal(obtenido, esperado), f"[{nombre_textura}] la tabla no coincide con skimage en {(obtenido != esperado).sum()} pixeles"
  print(f"OK [{nombre_textura}] etiquetas de 0 a {obtenido.max()}, coincide con skimage")

for nombre_textura in ['roto 1', 'fibroso 2', 'moteado']:
  verificar_tabla_nri_contra_skimage(nombre_textura)

#### **Ejemplos**

Veamos algunos patrones. Muevan el código de $0$ a $255$ y fijense cómo se ve el patrón en el círculo (naranja es $1$, blanco es $0$), cuántas transiciones tiene y a qué casillero va.

In [ ]:
def dibujar_patron(ax, patron_bits, titulo):
  # Dibuja los P vecinos sobre el circulo: naranja si el bit es 1 y blanco si es 0
  P = len(patron_bits)
  angulos = 2 * np.pi * np.arange(P) / P
  columnas = np.cos(angulos)
  filas = -np.sin(angulos)

  ax.add_patch(plt.Circle((0, 0), 1, fill=False, linestyle="--", color="gray"))
  for posicion in range(P):
    ax.scatter(columnas[posicion], filas[posicion], s=900, color="orange" if patron_bits[posicion] else "white", edgecolors="black", zorder=3)
    ax.text(columnas[posicion], filas[posicion], posicion, ha="center", va="center", fontsize=11, zorder=4)

  ax.set_xlim(-1.4, 1.4)
  ax.set_ylim(1.4, -1.4)
  ax.set_aspect("equal")
  ax.set_title(titulo)
  ax.axis("off")

def explorar_patron(codigo_lbp):
  patron_bits = bits_de_codigo(codigo_lbp, 8)
  transiciones = contar_transiciones(patron_bits)
  es_uniforme = transiciones <= 2

  fig, ax = plt.subplots(figsize=(4, 4))
  dibujar_patron(ax, patron_bits, f"Codigo {codigo_lbp}: {''.join(str(bit) for bit in patron_bits[::-1])} (bits del 7 al 0)\n{transiciones} transiciones, {'uniforme' if es_uniforme else 'NO uniforme'}\ncasillero {etiqueta_nri_uniforme(patron_bits)} (de 0 a 58)")
  plt.show()

interact(explorar_patron, codigo_lbp=IntSlider(min=0, max=255, step=1, value=104, description='codigo'));

**¿Qué tanto de una textura real es uniforme?** Si fuera muy poco, no ganaríamos nada con agrupar el resto. Armamos una tabla con el `contar_transiciones` que acabamos de escribir y miramos qué fracción de los píxeles de cada textura cae en un patrón uniforme.

In [ ]:
ES_UNIFORME_8 = np.array([contar_transiciones(bits_de_codigo(codigo_lbp, 8)) <= 2 for codigo_lbp in range(256)])

fracciones_uniformes = {}
for nombre_textura in TEXTURAS:
  mapa_comun = local_binary_pattern(cargar_textura(nombre_textura), 8, 1, method="default").astype(int)
  fracciones_uniformes[nombre_textura] = ES_UNIFORME_8[mapa_comun].mean()

plt.figure(figsize=(10, 4))
plt.bar(list(fracciones_uniformes), [fraccion * 100 for fraccion in fracciones_uniformes.values()])
plt.axhline(100 * 58 / 256, color="red", linestyle="--", label="si los 256 patrones fueran igual de probables (23%)")
plt.ylim(0, 100)
plt.ylabel("% de pixeles con patron uniforme")
plt.legend()
plt.show()

print(f"Entre el {min(fracciones_uniformes.values()) * 100:.0f}% y el {max(fracciones_uniformes.values()) * 100:.0f}% de los pixeles de cada textura tienen un patron uniforme, aunque son solo 58 de los 256 codigos")

Juntemos todo en un solo experimento. A una textura le agregamos **ruido** (mezclamos de lugar un porcentaje de sus píxeles), la **ampliamos** y calculamos su mapa LBP uniforme. Comparamos tres mapas: la imagen original con $R=1$, la imagen ampliada con $R=1$ y la imagen ampliada con el $R$ que elijan. Mientras más baja la diferencia entre sus histogramas, más parecida es la textura que ve LBP.

In [ ]:
def mostrar_escala_ruido_y_uniformes(textura, ruido, zoom, R):
  # Con la imagen original con ruido, la imagen ampliada y los mapas LBP uniformes de P=8 (59 etiquetas)
  original = mezclar_pixeles(cargar_textura(textura, 96), ruido)
  ampliada = util.img_as_ubyte(transform.resize(original, (96 * zoom, 96 * zoom), order=1, anti_aliasing=False))

  mapa_original = local_binary_pattern(original, 8, 1, method="nri_uniform").astype(int)
  mapa_ampliada_r1 = local_binary_pattern(ampliada, 8, 1, method="nri_uniform").astype(int)
  mapa_ampliada = local_binary_pattern(ampliada, 8, R, method="nri_uniform").astype(int)

  histograma_original = np.bincount(mapa_original.ravel(), minlength=59) / mapa_original.size
  histograma_ampliada_r1 = np.bincount(mapa_ampliada_r1.ravel(), minlength=59) / mapa_ampliada_r1.size
  histograma_ampliada = np.bincount(mapa_ampliada.ravel(), minlength=59) / mapa_ampliada.size
  diferencia_r1 = np.abs(histograma_original - histograma_ampliada_r1).sum()
  diferencia = np.abs(histograma_original - histograma_ampliada).sum()

  fig, ax = plt.subplots(3, 3, figsize=(15, 13))

  imagenes = [(cargar_textura(textura, 96), "Original sin ruido"), (original, f"Original con {ruido}% de ruido"), (ampliada, f"Original con ruido ampliada x{zoom}")]
  for eje, (imagen, titulo) in zip(ax[0], imagenes):
    eje.imshow(imagen, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
    eje.set_title(titulo)
    eje.axis("off")

  mapas = [(mapa_original, "Original, R=1"), (mapa_ampliada_r1, f"Ampliada x{zoom}, R=1"), (mapa_ampliada, f"Ampliada x{zoom}, R={R}")]
  for eje, (mapa, titulo) in zip(ax[1], mapas):
    eje.imshow(mapa, cmap="gray", vmin=0, vmax=58)
    eje.set_title(f"Mapa LBP uniforme: {titulo}")
    eje.axis("off")

  for eje, histograma, color, titulo, diferencia_mostrada in [(ax[2, 0], histograma_ampliada_r1, "red", "ampliada, R=1", diferencia_r1), (ax[2, 1], histograma_ampliada, "blue", f"ampliada, R={R}", diferencia)]:
    eje.bar(np.arange(59), histograma_original, color="black", alpha=0.5, label="original, R=1")
    eje.step(np.arange(59), histograma, where="mid", color=color, label=titulo)
    eje.set_title(f"Histogramas (diferencia {diferencia_mostrada:.2f})")
    eje.set_xlabel("etiqueta uniforme")
    eje.legend()

  barras = ax[2, 2].bar([f"ampliada, R=1", f"ampliada, R={R}"], [diferencia_r1, diferencia], color=["red", "blue"])
  ax[2, 2].bar_label(barras, fmt="%.2f")
  ax[2, 2].set_ylim(0, 2)
  ax[2, 2].set_title("Diferencia contra la original con R=1")

  plt.suptitle(f"{textura} con {ruido}% de ruido")
  plt.tight_layout()
  plt.show()

interact(
  mostrar_escala_ruido_y_uniformes,
  textura=widgets.Dropdown(options=list(TEXTURAS), value='roto 1', description='textura'),
  ruido=IntSlider(min=0, max=100, step=10, value=0, description='% ruido', continuous_update=False),
  zoom=IntSlider(min=1, max=4, step=1, value=3, description='zoom', continuous_update=False),
  R=IntSlider(min=1, max=8, step=1, value=3, description='R', continuous_update=False),
);

### **Invarianza a la rotación**

Imaginen que sacan dos fotos de la misma tela, una con la cámara rotada $90°$ respecto de la otra. Los bordes de la tela que antes eran horizontales ahora son verticales. Para una persona es la misma textura, pero los códigos cambian: un borde que antes tenía los unos en los vecinos $0$, $1$ y $2$, ahora los tiene en los $2$, $3$ y $4$. Son patrones distintos, con bines distintos, y los histogramas ya no se parecen.

La solución es considerar **iguales a todos los patrones que sean una rotación del círculo uno del otro**. Entre los patrones uniformes, todas las rotaciones de un patrón comparten algo: la **cantidad de unos**. Entonces

- si el patrón es uniforme, su etiqueta es la **cantidad de unos** ($0$ a $P$),
- si no, su etiqueta es $P+1$.

Quedan $P+2$ casilleros: **10 para $P=8$**. Esta variante se llama `riu2` (*rotation invariant uniform*, con $U \le 2$) y en skimage es `method="uniform"`. Las etiquetas además tienen un significado: $0$ es un máximo aislado (todos los vecinos son menores), $P$ es un mínimo o una zona homogenea, y las del medio son bordes y esquinas (cerca de $P/2$, un borde recto).

### ***Ejercicio***: Etiqueta invariante a la rotación

- Ejercicio: implementar
    - etiqueta_riu2

In [ ]:
def etiqueta_riu2(patron_bits):
  # Etiqueta del patron: cantidad de unos si es uniforme, P + 1 si no lo es
  # COMPLETAR
  pass

### Tests (riu2)

In [ ]:
# nombre: (patron de 8 bits, etiqueta esperada)
ETIQUETAS_RIU2_ESPERADAS = {
  'todos_ceros':          ([0, 0, 0, 0, 0, 0, 0, 0], 0),
  'todos_unos':           ([1, 1, 1, 1, 1, 1, 1, 1], 8),
  'un_solo_uno':          ([0, 0, 0, 1, 0, 0, 0, 0], 1),
  'borde':                ([0, 0, 0, 1, 1, 1, 0, 0], 3),
  'unos_dan_la_vuelta':   ([1, 1, 0, 0, 0, 0, 0, 1], 3),
  'no_uniforme':          ([0, 1, 0, 1, 0, 0, 0, 0], 9),  # P + 1
  'ejemplo_de_la_teoria': ([0, 0, 0, 1, 0, 1, 1, 0], 9),
}

def verificar_riu2(nombre):
  patron_bits, esperada = ETIQUETAS_RIU2_ESPERADAS[nombre]
  obtenida = etiqueta_riu2(patron_bits)

  assert obtenida == esperada, f"[{nombre}] se esperaba la etiqueta {esperada} pero se obtuvo {obtenida}"
  print(f"OK [{nombre}] etiqueta = {obtenida}")

def verificar_riu2_invariante_a_rotaciones():
  patron_bits = [0, 0, 1, 1, 1, 0, 0, 0]
  rotaciones = [np.roll(patron_bits, desplazamiento).tolist() for desplazamiento in range(8)]
  etiquetas_riu2 = {etiqueta_riu2(rotacion) for rotacion in rotaciones}
  etiquetas_nri = {etiqueta_nri_uniforme(rotacion) for rotacion in rotaciones}

  assert etiquetas_riu2 == {3}, f"las 8 rotaciones tienen que dar la misma etiqueta pero dieron {sorted(etiquetas_riu2)}"
  print(f"OK [rotaciones] las 8 rotaciones dan una sola etiqueta (con nri_uniforme darian {len(etiquetas_nri)} distintas)")

def verificar_riu2_contra_skimage(nombre_textura):
  imagen = cargar_textura(nombre_textura)
  mapa_comun = local_binary_pattern(imagen, 8, 1, method="default").astype(int)
  esperado = local_binary_pattern(imagen, 8, 1, method="uniform")
  obtenido = tabla_de_etiquetas(8, etiqueta_riu2)[mapa_comun]

  assert np.array_equal(obtenido, esperado), f"[{nombre_textura}] la tabla no coincide con skimage en {(obtenido != esperado).sum()} pixeles"
  print(f"OK [{nombre_textura}] etiquetas de 0 a {obtenido.max()}, coincide con skimage")

### Verificacion riu2

In [ ]:
for nombre in ETIQUETAS_RIU2_ESPERADAS:
  verificar_riu2(nombre)

verificar_riu2_invariante_a_rotaciones()
for nombre_textura in ['roto 1', 'fibroso 2', 'moteado']:
  verificar_riu2_contra_skimage(nombre_textura)

#### **Ejemplos**

Primero el patrón. Elijan un código y roten el círculo: con `nri_uniforme` cada rotación cae en un casillero distinto, con `riu2` todas caen en el mismo.

In [ ]:
def explorar_rotacion_de_patron(codigo_lbp, rotacion):
  patron_bits = bits_de_codigo(codigo_lbp, 8)
  patron_rotado = np.roll(patron_bits, rotacion).tolist()

  fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
  dibujar_patron(ax[0], patron_bits, f"Original\nnri_uniforme: {etiqueta_nri_uniforme(patron_bits)}   riu2: {etiqueta_riu2(patron_bits)}")
  dibujar_patron(ax[1], patron_rotado, f"Rotado {rotacion * 45} grados\nnri_uniforme: {etiqueta_nri_uniforme(patron_rotado)}   riu2: {etiqueta_riu2(patron_rotado)}")
  plt.tight_layout()
  plt.show()

interact(
  explorar_rotacion_de_patron,
  codigo_lbp=IntSlider(min=0, max=255, step=1, value=28, description='codigo'),
  rotacion=IntSlider(min=0, max=7, step=1, value=2, description='rotacion'),
);

Ahora lo mismo con una **imagen**. Rotamos una textura y comparamos el histograma de la original contra el de la rotada, con `nri_uniforme` (59 casilleros) y con `riu2` (10 casilleros). Cuando el ángulo es múltiplo de $90°$ la rotación de la imagen no pierde información y `riu2` debería dar casi la misma distribución. Con otros ángulos la rotación **interpola** y suaviza la imagen, y eso por sí solo ya cambia los histogramas.

In [ ]:
def recortar_centro(imagen, lado):
  fila_inicio = (imagen.shape[0] - lado) // 2
  columna_inicio = (imagen.shape[1] - lado) // 2
  return imagen[fila_inicio:fila_inicio + lado, columna_inicio:columna_inicio + lado]

def histograma_de_etiquetas(imagen, metodo, cantidad_de_etiquetas):
  mapa_etiquetas = local_binary_pattern(imagen, 8, 1, method=metodo).astype(int)
  return np.bincount(mapa_etiquetas.ravel(), minlength=cantidad_de_etiquetas) / mapa_etiquetas.size

def explorar_rotacion_de_imagen(textura, angulo):
  # Rotamos la textura completa y despues nos quedamos con el centro, para que no entren las esquinas negras
  textura_completa = cargar_textura(textura, 128)
  original = recortar_centro(textura_completa, 80)
  rotada = recortar_centro(np.round(transform.rotate(textura_completa, angulo, order=1, preserve_range=True)).astype(np.uint8), 80)

  histogramas = {'nri_uniform': 59, 'uniform': 10}
  diferencias = {metodo: np.abs(histograma_de_etiquetas(original, metodo, cantidad) - histograma_de_etiquetas(rotada, metodo, cantidad)).sum() for metodo, cantidad in histogramas.items()}

  fig, ax = plt.subplots(1, 4, figsize=(16, 4))

  ax[0].imshow(original, cmap="gray", vmin=0, vmax=255)
  ax[0].set_title("Original")
  ax[0].axis("off")

  ax[1].imshow(rotada, cmap="gray", vmin=0, vmax=255)
  ax[1].set_title(f"Rotada {angulo} grados")
  ax[1].axis("off")

  for eje, (metodo, cantidad) in zip(ax[2:], histogramas.items()):
    eje.bar(np.arange(cantidad) - 0.2, histograma_de_etiquetas(original, metodo, cantidad), width=0.4, label="original")
    eje.bar(np.arange(cantidad) + 0.2, histograma_de_etiquetas(rotada, metodo, cantidad), width=0.4, label="rotada")
    eje.set_title(f"{'nri_uniform (59)' if metodo == 'nri_uniform' else 'riu2 (10)'}: diferencia {diferencias[metodo]:.3f}")
    eje.legend()

  plt.tight_layout()
  plt.show()

interact(
  explorar_rotacion_de_imagen,
  textura=widgets.Dropdown(options=list(TEXTURAS), value='fibroso 2', description='textura'),
  angulo=IntSlider(min=0, max=90, step=15, value=90, description='angulo'),
);

Repitamos el experimento anterior, ahora con el LBP invariante a rotación. A la textura con ruido le sumamos un slider de **ángulo**: la rotamos, la ampliamos y comparamos los mapas `riu2` de la imagen original con $R=1$, de la imagen transformada con $R=1$ y de la imagen transformada con el $R$ que elijan.

In [ ]:
def mostrar_rotacion_escala_ruido_y_riu2(textura, ruido, zoom, angulo, R):
  # Mismo experimento que antes pero con el LBP invariante a rotacion de P=8 (10 etiquetas)
  completa = mezclar_pixeles(cargar_textura(textura, 128), ruido)
  original = recortar_centro(completa, 80)
  rotada = recortar_centro(np.round(transform.rotate(completa, angulo, order=1, preserve_range=True)).astype(np.uint8), 80)
  transformada = util.img_as_ubyte(transform.resize(rotada, (80 * zoom, 80 * zoom), order=1, anti_aliasing=False))

  mapa_original = local_binary_pattern(original, 8, 1, method="uniform").astype(int)
  mapa_transformada_r1 = local_binary_pattern(transformada, 8, 1, method="uniform").astype(int)
  mapa_transformada = local_binary_pattern(transformada, 8, R, method="uniform").astype(int)

  histograma_original = np.bincount(mapa_original.ravel(), minlength=10) / mapa_original.size
  histograma_transformada_r1 = np.bincount(mapa_transformada_r1.ravel(), minlength=10) / mapa_transformada_r1.size
  histograma_transformada = np.bincount(mapa_transformada.ravel(), minlength=10) / mapa_transformada.size
  diferencia_r1 = np.abs(histograma_original - histograma_transformada_r1).sum()
  diferencia = np.abs(histograma_original - histograma_transformada).sum()

  fig, ax = plt.subplots(3, 3, figsize=(15, 13))

  imagenes = [(recortar_centro(cargar_textura(textura, 128), 80), "Original sin ruido"), (original, f"Original con {ruido}% de ruido"), (transformada, f"Con ruido, rotada {angulo} grados y ampliada x{zoom}")]
  for eje, (imagen, titulo) in zip(ax[0], imagenes):
    eje.imshow(imagen, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
    eje.set_title(titulo)
    eje.axis("off")

  mapas = [(mapa_original, "Original, R=1"), (mapa_transformada_r1, f"Transformada, R=1"), (mapa_transformada, f"Transformada, R={R}")]
  for eje, (mapa, titulo) in zip(ax[1], mapas):
    eje.imshow(mapa, cmap="gray", vmin=0, vmax=9)
    eje.set_title(f"Mapa LBP riu2: {titulo}")
    eje.axis("off")

  for eje, histograma, color, titulo, diferencia_mostrada in [(ax[2, 0], histograma_transformada_r1, "red", "transformada, R=1", diferencia_r1), (ax[2, 1], histograma_transformada, "blue", f"transformada, R={R}", diferencia)]:
    eje.bar(np.arange(10), histograma_original, color="black", alpha=0.5, label="original, R=1")
    eje.step(np.arange(10), histograma, where="mid", color=color, label=titulo)
    eje.set_title(f"Histogramas (diferencia {diferencia_mostrada:.2f})")
    eje.set_xlabel("etiqueta riu2")
    eje.legend()

  barras = ax[2, 2].bar(["transformada, R=1", f"transformada, R={R}"], [diferencia_r1, diferencia], color=["red", "blue"])
  ax[2, 2].bar_label(barras, fmt="%.2f")
  ax[2, 2].set_ylim(0, 2)
  ax[2, 2].set_title("Diferencia contra la original con R=1")

  plt.suptitle(f"{textura} con {ruido}% de ruido")
  plt.tight_layout()
  plt.show()

interact(
  mostrar_rotacion_escala_ruido_y_riu2,
  textura=widgets.Dropdown(options=list(TEXTURAS), value='fibroso 2', description='textura'),
  ruido=IntSlider(min=0, max=100, step=10, value=0, description='% ruido', continuous_update=False),
  zoom=IntSlider(min=1, max=4, step=1, value=2, description='zoom', continuous_update=False),
  angulo=IntSlider(min=0, max=90, step=15, value=90, description='angulo', continuous_update=False),
  R=IntSlider(min=1, max=8, step=1, value=2, description='R', continuous_update=False),
);

### **Del mapa LBP al descriptor**

Un mapa LBP todavía no es un descriptor: es una imagen de etiquetas. Lo que caracteriza a la textura es **cuántas veces aparece cada etiqueta**, es decir, el **histograma** del mapa. Como no depende de dónde esté cada píxel, el histograma tiene la **invarianza traslacional** que queremos: la misma textura corrida de lugar da el mismo descriptor.

Esta misma invarianza tiene un costo, ya que perdemos **toda** la información espacial. Para una textura homogénea (una pared) no importa, pero para un objeto (una cara) sí: una boca arriba de una nariz y una nariz arriba de una boca tienen el mismo histograma. Una forma de quedarnos con lo mejor de los dos mundos es **dividir la imagen en una grilla de celdas**, calcular un histograma por celda y **concatenarlos**: dentro de cada celda hay invarianza traslacional pero entre celdas se conserva el orden.

### ***Ejercicio***: Histograma y descriptor por celdas

- Ejercicio: implementar
    - histograma_normalizado
    - descriptor_por_celdas

Ayudas:

- `histograma_normalizado` devuelve la **fracción** de píxeles que cae en cada etiqueta (suma $1$), así dos imágenes de distinto tamaño se pueden comparar. `np.bincount` con `minlength` es lo que necesitan.
- `descriptor_por_celdas` devuelve una matriz con **una fila por celda**, recorriendo las celdas por filas, y una columna por etiqueta. Si la imagen no se divide en celdas exactas, ignoren el sobrante del borde derecho e inferior.

In [ ]:
def histograma_normalizado(mapa_etiquetas, cantidad_de_etiquetas):
  # Fraccion de pixeles de cada etiqueta (de 0 a cantidad_de_etiquetas - 1)
  # COMPLETAR
  pass

In [ ]:
def descriptor_por_celdas(mapa_etiquetas, celdas_por_fila, celdas_por_columna, cantidad_de_etiquetas):
  # Una fila por celda (recorridas por filas), una columna por etiqueta
  alto_celda = mapa_etiquetas.shape[0] // celdas_por_fila
  ancho_celda = mapa_etiquetas.shape[1] // celdas_por_columna

  # COMPLETAR

### Tests (descriptor por celdas)

In [ ]:
# nombre: (mapa de etiquetas, cantidad de etiquetas, histograma esperado)
HISTOGRAMAS_ESPERADOS = {
  'una_etiqueta':     (np.array([[2, 2], [2, 2]]),         3, np.array([0, 0, 1])),
  'mezcla':           (np.array([[0, 1], [1, 1]]),         3, np.array([0.25, 0.75, 0])),
  'etiqueta_sin_uso': (np.array([[0, 0, 3], [0, 0, 3]]),   4, np.array([4 / 6, 0, 0, 2 / 6])),
}

# Mapa de 4x4 dividido en 2x2 celdas. Las celdas, por filas, son: todo 0, todo 1, todo 2 y [0, 1, 1, 1]
MAPA_PARA_CELDAS = np.array([[0, 0, 1, 1], [0, 0, 1, 1], [2, 2, 0, 1], [2, 2, 1, 1]])
DESCRIPTOR_CELDAS_ESPERADO = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1], [0.25, 0.75, 0]])

def verificar_histograma(nombre):
  mapa_etiquetas, cantidad_de_etiquetas, esperado = HISTOGRAMAS_ESPERADOS[nombre]
  obtenido = histograma_normalizado(mapa_etiquetas, cantidad_de_etiquetas)

  assert np.allclose(obtenido, esperado), f"[{nombre}] se esperaba {esperado.tolist()} pero se obtuvo {np.round(obtenido, 3).tolist()}"
  print(f"OK [{nombre}] histograma = {np.round(obtenido, 3).tolist()}")

def verificar_descriptor_por_celdas():
  obtenido = descriptor_por_celdas(MAPA_PARA_CELDAS, 2, 2, 3)

  assert obtenido.shape == DESCRIPTOR_CELDAS_ESPERADO.shape, f"[celdas] se esperaba una matriz de forma {DESCRIPTOR_CELDAS_ESPERADO.shape} pero se obtuvo {obtenido.shape}"
  assert np.allclose(obtenido, DESCRIPTOR_CELDAS_ESPERADO), f"[celdas] se esperaba {DESCRIPTOR_CELDAS_ESPERADO.tolist()} pero se obtuvo {obtenido.tolist()}"
  print(f"OK [celdas] descriptor = {obtenido.tolist()}")

def verificar_descriptor_sobrante():
  # 80 no se divide en 7 celdas exactas: cada celda es de 11x11 y sobran 3 filas y 3 columnas
  mapa_etiquetas = np.random.default_rng(0).integers(0, 59, (80, 80))
  obtenido = descriptor_por_celdas(mapa_etiquetas, 7, 7, 59)

  assert obtenido.shape == (49, 59), f"[sobrante] se esperaba una matriz de forma (49, 59) pero se obtuvo {obtenido.shape}"
  assert np.allclose(obtenido.sum(axis=1), 1), "[sobrante] el histograma de cada celda tiene que sumar 1"
  esperado_primera_celda = histograma_normalizado(mapa_etiquetas[:11, :11], 59)
  assert np.allclose(obtenido[0], esperado_primera_celda), "[sobrante] la primera celda tiene que ser la de arriba a la izquierda, de 11x11"
  print(f"OK [sobrante] 49 celdas de 59 etiquetas, cada una suma 1")

### Verificacion histograma

In [ ]:
for nombre in HISTOGRAMAS_ESPERADOS:
  verificar_histograma(nombre)

### Verificacion descriptor por celdas

In [ ]:
verificar_descriptor_por_celdas()
verificar_descriptor_sobrante()

#### **Ejemplo: el histograma LBP como descriptor de textura**

Veamos si el histograma `riu2` alcanza para distinguir texturas. Dividimos cada textura en una grilla de $6 \times 6$ ventanas de $32 \times 32$ píxeles y calculamos el histograma de cada ventana. Después elegimos **dos bines** del histograma (dos etiquetas) y graficamos cada ventana como un punto: $x$ es la fracción de píxeles con la primera etiqueta e $y$ la de la segunda. Cada color es un tipo de textura y cada forma, una foto distinta.

Si las ventanas de la misma textura forman una nube y las de texturas distintas forman nubes separadas, entonces esas dos etiquetas alcanzan para distinguirlas. Prueben distintos pares: algunos separan bien un par de texturas y mezclan a las demás, y ninguno las separa a todas a la vez.

In [ ]:
CLASES_TEXTURA = {
  'manchado 1': 'manchado', 'manchado 2': 'manchado', 'roto 1': 'roto', 'roto 2': 'roto',
  'fibroso 1': 'fibroso', 'fibroso 2': 'fibroso', 'moteado': 'moteado', 'cristalino': 'cristalino',
}
COLORES_CLASES = {'manchado': 'tab:blue', 'roto': 'tab:red', 'fibroso': 'tab:green', 'moteado': 'tab:orange', 'cristalino': 'tab:purple'}

# Un histograma riu2 (10 etiquetas) por cada ventana de 32x32 de cada textura
DESCRIPTORES_TEXTURAS = {
  nombre: descriptor_por_celdas(local_binary_pattern(cargar_textura(nombre), 8, 1, method="uniform").astype(int), 6, 6, 10)
  for nombre in TEXTURAS
}

def graficar_scatter_de_etiquetas(etiqueta_x, etiqueta_y):
  plt.figure(figsize=(7, 6))
  clases_ya_en_la_leyenda = set()
  for nombre, descriptor in DESCRIPTORES_TEXTURAS.items():
    clase = CLASES_TEXTURA[nombre]
    marcador = "o" if nombre.endswith("1") or clase in ("moteado", "cristalino") else "^"
    plt.scatter(descriptor[:, etiqueta_x], descriptor[:, etiqueta_y], color=COLORES_CLASES[clase], marker=marcador, alpha=0.7, label=None if clase in clases_ya_en_la_leyenda else clase)
    clases_ya_en_la_leyenda.add(clase)

  plt.xlabel(f"fraccion de pixeles con la etiqueta {etiqueta_x}")
  plt.ylabel(f"fraccion de pixeles con la etiqueta {etiqueta_y}")
  plt.title("Una ventana de 32x32 por punto")
  plt.grid(True)
  plt.legend()
  plt.show()

interact(
  graficar_scatter_de_etiquetas,
  etiqueta_x=IntSlider(min=0, max=9, step=1, value=0, description='etiqueta x'),
  etiqueta_y=IntSlider(min=0, max=9, step=1, value=4, description='etiqueta y'),
);

Con las etiquetas correctas, un descriptor de solo **dos números** ya separa bastante bien algunas texturas, y con el histograma completo (10 números, o 59 con `nri_uniform`) tenemos mucha más información. Esa es la gracia de LBP: es un descriptor chico, rápido de calcular y invariante a la iluminación, rotacion y escala.

### **Reconocimiento de rostros con LBP**

Aprovechando que LBP es robusto a los cambios de iluminación y que el descriptor por celdas conserva el orden de las partes de la cara, podemos usarlo para **reconocer rostros**. Vamos a seguir una versión simplificada del paper [Face Recognition with Local Binary Patterns](https://link.springer.com/chapter/10.1007/978-3-540-24670-1_36) (Ahonen, Hadid y Pietikäinen, 2004). Su receta es:

1. Calcular el mapa LBP uniforme ($P=8$, $R=2$) de la cara.
2. Dividir la cara en una grilla de $7 \times 7$ celdas y calcular el histograma de cada una. El descriptor de la cara son las $49$ celdas.
3. Comparar dos caras con la distancia **chi-cuadrado ponderada**, celda por celda.
4. Identificar a una persona como la **más cercana** de las que conocemos.

Fijense que acá usamos `nri_uniforme` (59 casilleros) y **no** `riu2`. En una cara no combiene unificar toda la informacion ignorando su orientacion, por ejemplo un borde horizontal es una ceja y uno vertical puede ser el costado de la nariz.

Los datos son un subconjunto de **FERET** con $200$ personas y $7$ fotos de cada una. Miren las fotos de distintas personas: la $01$ es frontal y neutra, la $06$ es sonriendo, la $07$ tiene otra iluminación y en las fotos de la $02$ a la $05$ la cabeza está girada (mucho en la $02$ y la $05$, apenas en la $03$ y la $04$).

In [ ]:
def cargar_feret():
  # Arreglo de forma (200 personas, 7 fotos, 80, 80)
  return np.array([[io.imread(f"{CARPETA_FERET}/FERET-{persona:03d}/{foto:02d}.tif") for foto in range(1, 8)] for persona in range(1, 201)])

imagenes_feret = cargar_feret()
print(f"Cargadas {imagenes_feret.shape[0]} personas con {imagenes_feret.shape[1]} fotos de {imagenes_feret.shape[2]}x{imagenes_feret.shape[3]}")

def mostrar_persona(persona):
  fig, ax = plt.subplots(1, 7, figsize=(14, 2.5))
  for foto in range(7):
    ax[foto].imshow(imagenes_feret[persona - 1, foto], cmap="gray", vmin=0, vmax=255)
    ax[foto].set_title(f"foto 0{foto + 1}")
    ax[foto].axis("off")
  plt.tight_layout()
  plt.show()

interact(mostrar_persona, persona=IntSlider(min=1, max=200, step=1, value=1, description='persona'));

#### **Chi-cuadrado ponderado**

Para comparar dos histogramas $S$ y $M$ el paper usa la distancia **chi-cuadrado**, que le da más importancia a las diferencias en los casilleros chicos que en los grandes:

$$\chi^2(S, M) = \sum_i \frac{(S_i - M_i)^2}{S_i + M_i}$$

Con el descriptor por celdas la calculamos **celda por celda**: $\chi^2_j$ es la distancia entre los histogramas de la celda $j$. Y acá viene la idea importante del paper. **No todas las partes de la cara aportan lo mismo** para reconocer a una persona: los ojos, las cejas y la boca dicen mucho más que las mejillas o la frente. Entonces la distancia final **pondera** cada celda con un peso $w_j$:

$$\chi^2_w(S, M) = \sum_j w_j \chi^2_j(S, M) = \sum_j w_j \sum_i \frac{(S_{j,i} - M_{j,i})^2}{S_{j,i} + M_{j,i}}$$

Con todos los pesos iguales a $1$ es la distancia común, y con $w_j = 0$ una celda deja de importar.

### ***Ejercicio***: Chi-cuadrado ponderado

- Ejercicio: implementar
    - distancia_chi2_por_celda
    - distancia_chi2_ponderada
    - identificar

Ayudas:

- Los descriptores tienen forma `(celdas, etiquetas)`. `distancia_chi2_por_celda` devuelve **un valor por celda** (un vector con `celdas` elementos). Sumen sobre el **último eje** con `axis=-1`: así funciona también cuando `descriptor_a` trae varios descriptores a la vez, con forma `(n, celdas, etiquetas)`.
- Si ambos histogramas valen $0$ en un casillero la fracción es $0/0$. Sumenle `1e-10` al denominador.
- `identificar` recibe un **cubo de distancias** con forma `(probes, personas_conocidas, celdas)`: el $\chi^2_j$ de cada foto a identificar contra cada persona conocida, celda por celda. Tiene que devolver, para cada probe, el índice de la persona conocida más cercana según la distancia ponderada. Cambiar los pesos puede cambiar quién gana.

In [ ]:
def distancia_chi2_por_celda(descriptor_a, descriptor_b):
  # Chi-cuadrado de cada celda: un valor por celda. Admite varios descriptores en los primeros ejes
  # COMPLETAR
  pass

In [ ]:
def distancia_chi2_ponderada(descriptor_a, descriptor_b, pesos):
  # Chi-cuadrado ponderado: suma de los chi-cuadrado de cada celda por el peso de la celda
  # Los pesos pueden venir como una grilla (7, 7): hay un peso por celda, recorridas por filas
  # COMPLETAR
  pass

In [ ]:
def identificar(cubo_de_distancias, pesos):
  # Para cada probe devuelve el indice de la persona conocida mas cercana segun la distancia ponderada
  # COMPLETAR
  pass

### Tests (chi-cuadrado ponderado)

In [ ]:
# Dos descriptores de 2 celdas con 2 etiquetas. La primera celda es totalmente distinta y la segunda es igual
DESCRIPTOR_A = np.array([[1.0, 0.0], [0.5, 0.5]])
DESCRIPTOR_B = np.array([[0.0, 1.0], [0.5, 0.5]])

# nombre: (pesos, distancia ponderada esperada). La celda 0 aporta chi2 = 1 + 1 = 2 y la celda 1 aporta 0
DISTANCIAS_PONDERADAS_ESPERADAS = {
  'pesos_uniformes':    (np.array([1, 1]), 2),
  'celda_0_pesa_triple': (np.array([3, 1]), 6),
  'celda_0_sin_peso':   (np.array([0, 5]), 0),
  'pesos_en_grilla':    (np.array([[3, 1]]), 6),  # los pesos pueden venir como una grilla: se recorren por filas
}

# El cubo tiene 2 probes, 3 personas conocidas y 2 celdas. Segun los pesos, cada probe se identifica distinto
CUBO_PARA_IDENTIFICAR = np.array([
  [[1, 5], [4, 0], [3, 3]],
  [[2, 3], [5, 2], [0, 9]],
])
IDENTIFICACIONES_ESPERADAS = {
  'pesos_uniformes':   (np.array([1, 1]), [1, 0]),
  'solo_celda_1':      (np.array([0, 1]), [1, 1]),
  'solo_celda_0':      (np.array([1, 0]), [0, 2]),
}

def verificar_chi2_por_celda():
  obtenida = distancia_chi2_por_celda(DESCRIPTOR_A, DESCRIPTOR_B)

  assert np.shape(obtenida) == (2,), f"[por celda] se esperaba un valor por celda (forma (2,)) pero se obtuvo la forma {np.shape(obtenida)}"
  assert np.allclose(obtenida, [2, 0]), f"[por celda] se esperaba [2, 0] pero se obtuvo {np.round(obtenida, 3).tolist()}"
  print(f"OK [por celda] chi2 = {np.round(obtenida, 3).tolist()}")

def verificar_chi2_con_varios_descriptores():
  varios = np.stack([DESCRIPTOR_A, DESCRIPTOR_B, DESCRIPTOR_A])
  obtenida = distancia_chi2_por_celda(varios, DESCRIPTOR_B)

  assert np.shape(obtenida) == (3, 2), f"[varios] se esperaba la forma (3, 2) pero se obtuvo {np.shape(obtenida)}"
  assert np.allclose(obtenida, [[2, 0], [0, 0], [2, 0]]), f"[varios] se esperaba [[2, 0], [0, 0], [2, 0]] pero se obtuvo {np.round(obtenida, 3).tolist()}"
  print("OK [varios] un chi2 por celda para cada descriptor")

def verificar_chi2_sin_division_por_cero():
  ceros = np.zeros((2, 3))
  obtenida = distancia_chi2_por_celda(ceros, ceros)

  assert np.all(np.isfinite(obtenida)) and np.allclose(obtenida, 0), "[ceros] dos histogramas vacios tienen que dar distancia 0 y no nan"
  print("OK [ceros] 0 / 0 no rompe la distancia")

def verificar_chi2_ponderada(nombre):
  pesos, esperada = DISTANCIAS_PONDERADAS_ESPERADAS[nombre]
  obtenida = distancia_chi2_ponderada(DESCRIPTOR_A, DESCRIPTOR_B, pesos)

  assert np.isclose(obtenida, esperada), f"[{nombre}] se esperaba {esperada} pero se obtuvo {obtenida}"
  print(f"OK [{nombre}] distancia = {obtenida:.3f}")

def verificar_identificar(nombre):
  pesos, esperadas = IDENTIFICACIONES_ESPERADAS[nombre]
  obtenidas = identificar(CUBO_PARA_IDENTIFICAR, pesos)

  assert np.array_equal(obtenidas, esperadas), f"[{nombre}] se esperaba {esperadas} pero se obtuvo {np.asarray(obtenidas).tolist()}"
  print(f"OK [{nombre}] identificaciones = {np.asarray(obtenidas).tolist()}")

### Verificacion chi-cuadrado

In [ ]:
verificar_chi2_por_celda()
verificar_chi2_con_varios_descriptores()
verificar_chi2_sin_division_por_cero()
for nombre in DISTANCIAS_PONDERADAS_ESPERADAS:
  verificar_chi2_ponderada(nombre)

### Verificacion identificar

In [ ]:
for nombre in IDENTIFICACIONES_ESPERADAS:
  verificar_identificar(nombre)

#### **Preparando la evaluación**

Para reconocer una cara necesitamos dos cosas: una **galería** con **una foto por persona** (las que el sistema conoce) y un **probe**, otra foto de una de esas personas que hay que identificar. Si el descriptor del probe queda más cerca del de su misma persona que del de las otras $199$, acertamos.

Cualquier foto puede hacer de galería o de probe. Para tener una evaluación estable promediamos **todas las parejas** entre las fotos $01$, $03$, $04$, $06$ y $07$: cada una hace de galería con las otras cuatro de probe, lo que da $5 \times 4 = 20$ parejas de $200$ identificaciones, es decir, $4000$ en total. Dejamos afuera la $02$ y la $05$ porque la cabeza está tan girada que cada celda termina mirando otra parte de la cara y el descriptor por celdas, que supone **caras alineadas**, ya no tiene sentido (al final de la sección pueden probar agregarlas).

Primero calculamos el descriptor de cada foto: el mapa LBP uniforme con $P=8$, $R=2$ y una grilla de $7 \times 7$ celdas, con $59$ etiquetas por celda. Como ya verificamos que nuestras implementaciones coinciden con las de skimage, usamos skimage para que sea rápido.

In [ ]:
P_ROSTROS = 8
R_ROSTROS = 2
CANTIDAD_ETIQUETAS_NRI = P_ROSTROS * (P_ROSTROS - 1) + 3  # 58 uniformes y uno para los no uniformes
GRILLA_ROSTROS = 7
FOTOS_EVALUADAS = [0, 2, 3, 5, 6]  # fotos 01, 03, 04, 06 y 07 (la numeracion arranca en 0)

### ***Ejercicio***: Descriptor de un rostro

Completar la función que arma el descriptor de una cara siguiendo la receta del paper: mapa LBP uniforme y concatenación de los histogramas de cada celda.

- Ejercicio: implementar
    - descriptor_de_rostro

Ayudas:

- Ya están dadas las constantes de arriba: $P=8$ vecinos, radio $R=2$ por defecto y una grilla de $7 \times 7$ celdas.
- El mapa LBP lo calcula skimage (ya verificamos que coincide con el nuestro): `local_binary_pattern` con `method="nri_uniform"`. Devuelve un mapa de la misma forma que la imagen.
- Después usen el `descriptor_por_celdas` que ya implementaron. La grilla tiene `celdas_por_lado` celdas por fila y por columna.
- El descriptor tiene que ser `float32`: alcanza y hace que las miles de distancias que vamos a calcular tarden la mitad.

In [ ]:
def descriptor_de_rostro(imagen, celdas_por_lado=GRILLA_ROSTROS, R=R_ROSTROS):
  # Devuelve una matriz de (celdas_por_lado ** 2) x CANTIDAD_ETIQUETAS_NRI, en float32
  # COMPLETAR
  pass

### Tests (descriptor de rostro)

In [ ]:
# nombre: (celdas por lado, forma esperada del descriptor)
FORMAS_ESPERADAS_ROSTRO = {
  'grilla_7': (7, (49, 59)),
  'grilla_4': (4, (16, 59)),
  'grilla_1': (1, (1, 59)),
}

# nombre: (fila de la celda, columna de la celda) dentro de la grilla
CELDAS_A_VERIFICAR = {
  'arriba_a_la_izquierda': (0, 0),
  'a_la_derecha_de_la_primera': (0, 1),
  'abajo_de_la_primera': (1, 0),
  'abajo_a_la_derecha': (6, 6),
}

ETIQUETA_TODOS_UNOS = P_ROSTROS * (P_ROSTROS - 1) + 1  # 57: todos los vecinos son mayores o iguales que el centro
IMAGEN_PLANA_ROSTRO = np.full((80, 80), 50, dtype=np.uint8)

def verificar_forma_del_descriptor(nombre):
  celdas_por_lado, esperada = FORMAS_ESPERADAS_ROSTRO[nombre]
  obtenido = descriptor_de_rostro(imagenes_feret[0, 0], celdas_por_lado)

  assert obtenido.shape == esperada, f"[{nombre}] se esperaba una matriz de forma {esperada} pero se obtuvo {obtenido.shape}"
  assert obtenido.dtype == np.float32, f"[{nombre}] se esperaba float32 pero se obtuvo {obtenido.dtype}"
  assert np.allclose(obtenido.sum(axis=1), 1, atol=1e-5), f"[{nombre}] el histograma de cada celda tiene que sumar 1"
  print(f"OK [{nombre}] descriptor de forma {obtenido.shape}")

def verificar_celda(nombre):
  # Las celdas se recorren por filas y cada una es de 11x11 (80 // 7)
  fila_celda, columna_celda = CELDAS_A_VERIFICAR[nombre]
  imagen = imagenes_feret[3, 5]
  mapa_etiquetas = local_binary_pattern(imagen, P_ROSTROS, R_ROSTROS, method="nri_uniform").astype(int)
  esperada = histograma_normalizado(mapa_etiquetas[fila_celda * 11:(fila_celda + 1) * 11, columna_celda * 11:(columna_celda + 1) * 11], CANTIDAD_ETIQUETAS_NRI)
  obtenida = descriptor_de_rostro(imagen)[fila_celda * GRILLA_ROSTROS + columna_celda]

  assert np.allclose(obtenida, esperada, atol=1e-6), f"[{nombre}] la fila {fila_celda * GRILLA_ROSTROS + columna_celda} del descriptor tiene que ser el histograma de la celda de la fila {fila_celda} y la columna {columna_celda} del mapa nri_uniform"
  print(f"OK [{nombre}] la celda ({fila_celda}, {columna_celda}) esta en la fila {fila_celda * GRILLA_ROSTROS + columna_celda} del descriptor")

def verificar_imagen_plana():
  # En una imagen plana todos los vecinos del interior son iguales al centro, asi que la etiqueta es siempre la de "todos unos".
  # Las celdas del borde quedan afuera porque skimage rellena con 0 fuera de la imagen
  interior = descriptor_de_rostro(IMAGEN_PLANA_ROSTRO).reshape(7, 7, -1)[1:-1, 1:-1]

  assert np.allclose(interior[:, :, ETIQUETA_TODOS_UNOS], 1), f"en una imagen plana las celdas del interior tienen que tener todos los pixeles en la etiqueta {ETIQUETA_TODOS_UNOS}"
  print(f"OK [plana] las celdas del interior tienen todos los pixeles en la etiqueta {ETIQUETA_TODOS_UNOS}")

def verificar_usa_el_radio():
  imagen = imagenes_feret[0, 0]

  assert not np.allclose(descriptor_de_rostro(imagen, R=1), descriptor_de_rostro(imagen, R=3)), "el descriptor tiene que cambiar con el radio R"
  print("OK [radio] el descriptor cambia con R")

### Verificacion descriptor de rostro

In [ ]:
for nombre in FORMAS_ESPERADAS_ROSTRO:
  verificar_forma_del_descriptor(nombre)

for nombre in CELDAS_A_VERIFICAR:
  verificar_celda(nombre)

verificar_imagen_plana()
verificar_usa_el_radio()

### ***Ejercicio***: Descriptores de una foto de las 200 personas

Completar la función que calcula, para una misma foto, el descriptor de las $200$ personas.

- Ejercicio: implementar
    - calcular_descriptores_de_foto

Ayudas:

- `imagenes_feret` tiene forma `(personas, fotos, 80, 80)`: la persona va primero y la foto después.
- El resultado tiene que tener un descriptor por persona, **en el orden de las personas**: la fila $i$ es el descriptor de la persona $i$. La forma es `(200, 49, 59)`.

In [ ]:
def calcular_descriptores_de_foto(foto):
  # Descriptores de la misma foto de las 200 personas: forma (200, 49, 59)
  # COMPLETAR
  pass

### Tests (descriptores de una foto)

In [ ]:
# nombre: foto (la numeracion arranca en 0)
FOTOS_A_VERIFICAR = {'foto_01': 0, 'foto_07': 6}
PERSONAS_A_VERIFICAR = [0, 57, 199]

def verificar_descriptores_de_foto(nombre):
  foto = FOTOS_A_VERIFICAR[nombre]
  obtenidos = calcular_descriptores_de_foto(foto)
  esperada = (200, GRILLA_ROSTROS * GRILLA_ROSTROS, CANTIDAD_ETIQUETAS_NRI)

  assert obtenidos.shape == esperada, f"[{nombre}] se esperaba un arreglo de forma {esperada} pero se obtuvo {obtenidos.shape}"
  for persona in PERSONAS_A_VERIFICAR:
    assert np.allclose(obtenidos[persona], descriptor_de_rostro(imagenes_feret[persona, foto])), f"[{nombre}] la fila {persona} tiene que ser el descriptor de la foto {foto + 1} de la persona {persona}"
  print(f"OK [{nombre}] 200 descriptores de forma {obtenidos.shape[1:]}, en el orden de las personas")

### Verificacion descriptores de una foto

In [ ]:
for nombre in FOTOS_A_VERIFICAR:
  verificar_descriptores_de_foto(nombre)

In [ ]:
inicio = time.perf_counter()
DESCRIPTORES_ROSTROS = {foto: calcular_descriptores_de_foto(foto) for foto in FOTOS_EVALUADAS}
print(f"Descriptores de {len(FOTOS_EVALUADAS)} fotos de 200 personas calculados en {time.perf_counter() - inicio:.1f} s")
print(f"Cada rostro es una matriz de {DESCRIPTORES_ROSTROS[0].shape[1:]}: {GRILLA_ROSTROS * GRILLA_ROSTROS} celdas de {CANTIDAD_ETIQUETAS_NRI} etiquetas")

Las distancias celda por celda **no dependen de los pesos**, así que las calculamos una sola vez para cada pareja (galería, probe) y las guardamos en un cubo. Después probar un mapa de pesos nuevo es solo una multiplicación de matrices.

### ***Ejercicio***: Cubo de distancias y precisión

Completar las funciones que arman el cubo de distancias y miden la precisión del reconocimiento.

- Ejercicio: implementar
    - cubo_de_distancias
    - calcular_precision

Ayudas:

- `cubo_de_distancias` devuelve el chi-cuadrado celda por celda de cada probe contra cada persona de la galería, con forma `(probes, galeria, celdas)`. Recuerden que `distancia_chi2_por_celda` admite varios descriptores a la vez: ¿qué pasa si le pasan **todos los probes** contra **un solo** descriptor de la galería? Armen el cubo con eso.
- `calcular_precision` recibe un cubo, por ejemplo `CUBOS_ROSTROS[(galeria, probe)]`, y los pesos. El probe número $i$ es de la persona número $i$: se acierta cuando `identificar` devuelve $i$. Devuelve la **fracción de aciertos**.

In [ ]:
def cubo_de_distancias(descriptores_probe, descriptores_galeria):
  # chi2 celda por celda de cada probe contra cada persona de la galeria: forma (probes, galeria, celdas)
  # COMPLETAR
  pass

def calcular_precision(cubo, pesos):
  # El probe numero i es de la persona numero i: acertamos cuando identificamos a i
  # COMPLETAR
  pass

### Tests (cubo de distancias y precisión)

In [ ]:
# Dos probes y tres personas de la galeria, con descriptores de 2 celdas y 2 etiquetas (los de los tests de chi-cuadrado)
# El probe 0 es igual a las personas 0 y 2 y el probe 1 es igual a la persona 1: cada chi2 es 0 si son iguales y [2, 0] si no
DESCRIPTORES_PROBE_PRUEBA = np.stack([DESCRIPTOR_A, DESCRIPTOR_B])
DESCRIPTORES_GALERIA_PRUEBA = np.stack([DESCRIPTOR_A, DESCRIPTOR_B, DESCRIPTOR_A])
CUBO_ESPERADO = np.array([[[0, 0], [2, 0], [0, 0]], [[2, 0], [0, 0], [2, 0]]])

# El cubo donde cada probe esta mucho mas cerca de su persona
CUBO_TODAS_CORRECTAS = np.array([[[0, 0], [5, 5]], [[5, 5], [0, 0]]])

# nombre: (cubo, pesos, precision esperada). Con CUBO_PARA_IDENTIFICAR las identificaciones eran [1, 0], [0, 2] y [1, 1]
PRECISIONES_ESPERADAS = {
  'todas_correctas': (CUBO_TODAS_CORRECTAS,   np.array([1, 1]), 1.0),
  'pesos_uniformes': (CUBO_PARA_IDENTIFICAR,  np.array([1, 1]), 0.0),
  'solo_celda_0':    (CUBO_PARA_IDENTIFICAR,  np.array([1, 0]), 0.5),
  'solo_celda_1':    (CUBO_PARA_IDENTIFICAR,  np.array([0, 1]), 0.5),
}

def verificar_cubo_chico():
  obtenido = cubo_de_distancias(DESCRIPTORES_PROBE_PRUEBA, DESCRIPTORES_GALERIA_PRUEBA)

  assert obtenido.shape == (2, 3, 2), f"[cubo chico] se esperaba la forma (2, 3, 2) (probes, galeria, celdas) pero se obtuvo {obtenido.shape}"
  assert np.allclose(obtenido, CUBO_ESPERADO), f"[cubo chico] se esperaba {CUBO_ESPERADO.tolist()} pero se obtuvo {np.round(obtenido, 3).tolist()}"
  print("OK [cubo chico] forma (2, 3, 2) y las distancias esperadas")

def verificar_cubo_de_rostros():
  probes = DESCRIPTORES_ROSTROS[2][:5]
  galeria = DESCRIPTORES_ROSTROS[0][:7]
  obtenido = cubo_de_distancias(probes, galeria)

  assert obtenido.shape == (5, 7, GRILLA_ROSTROS * GRILLA_ROSTROS), f"[rostros] se esperaba la forma (5, 7, 49) pero se obtuvo {obtenido.shape}"
  assert np.allclose(obtenido[3, 4], distancia_chi2_por_celda(probes[3], galeria[4]), atol=1e-5), "[rostros] cubo[3, 4] tiene que ser el chi2 celda por celda del probe 3 contra la persona 4"
  print(f"OK [rostros] 5 probes contra 7 personas: forma {obtenido.shape}")

def verificar_precision(nombre):
  cubo, pesos, esperada = PRECISIONES_ESPERADAS[nombre]
  obtenida = calcular_precision(cubo, pesos)

  assert np.isclose(obtenida, esperada), f"[{nombre}] se esperaba una precision de {esperada} pero se obtuvo {obtenida}"
  print(f"OK [{nombre}] precision = {obtenida}")

### Verificacion cubo de distancias y precisión

In [ ]:
verificar_cubo_chico()
verificar_cubo_de_rostros()
for nombre in PRECISIONES_ESPERADAS:
  verificar_precision(nombre)

In [ ]:
inicio = time.perf_counter()
CUBOS_ROSTROS = {
  (galeria, probe): cubo_de_distancias(DESCRIPTORES_ROSTROS[probe], DESCRIPTORES_ROSTROS[galeria])
  for galeria in FOTOS_EVALUADAS for probe in FOTOS_EVALUADAS if galeria != probe
}
print(f"{len(CUBOS_ROSTROS)} cubos de distancias de forma {CUBOS_ROSTROS[(0, 2)].shape} calculados en {time.perf_counter() - inicio:.1f} s")

def evaluar_pesos(pesos):
  # Matriz de precisiones: fila = foto de la galeria, columna = foto del probe. La diagonal no se evalua
  precisiones = np.full((len(FOTOS_EVALUADAS), len(FOTOS_EVALUADAS)), np.nan)
  for fila, galeria in enumerate(FOTOS_EVALUADAS):
    for columna, probe in enumerate(FOTOS_EVALUADAS):
      if galeria != probe: precisiones[fila, columna] = calcular_precision(CUBOS_ROSTROS[(galeria, probe)], pesos)

  return precisiones

#### **Los pesos del paper**

Los autores no eligieron los pesos al azar: probaron qué celdas aportan más para reconocer y armaron este mapa. Los pesos van de $0$ a $4$ y siguen una idea sencilla: **los ojos y las cejas valen mucho** ($4$), la boca y las esquinas de arriba valen el doble ($2$), la mayor parte de la cara vale $1$ y las zonas del costado, donde hay pelo, fondo y sombras que cambian de foto en foto, **no valen nada** ($0$).

Veamos el mapa sobre la cara promedio de las $200$ personas para ver qué zona de la cara cubre cada celda.

In [ ]:
PESOS_UNIFORMES = np.ones((GRILLA_ROSTROS, GRILLA_ROSTROS))

# Mapa de pesos del paper para una grilla de 7x7 celdas
PESOS_DEL_PAPER = np.array([
  [2, 1, 1, 1, 1, 1, 2],
  [2, 4, 4, 1, 4, 4, 2],  # ojos y cejas
  [1, 1, 1, 0, 1, 1, 1],
  [0, 1, 1, 0, 1, 1, 0],
  [0, 1, 1, 1, 1, 1, 0],
  [0, 1, 1, 2, 1, 1, 0],  # boca
  [0, 1, 1, 1, 1, 1, 0],
], dtype=float)

cara_promedio = imagenes_feret[:, 0].mean(axis=0)

def mostrar_pesos(pesos, titulo="Pesos"):
  fig, ax = plt.subplots(1, 2, figsize=(10, 5))
  lado_celda = cara_promedio.shape[0] // pesos.shape[0]

  ax[0].imshow(cara_promedio, cmap="gray")
  for posicion in np.arange(0, pesos.shape[0] + 1) * lado_celda - 0.5:
    ax[0].axhline(posicion, color="yellow", linewidth=0.8)
    ax[0].axvline(posicion, color="yellow", linewidth=0.8)
  ax[0].set_title("Cara promedio y la grilla de celdas")
  ax[0].axis("off")

  ax[1].imshow(pesos, cmap="gray", vmin=0, vmax=4)
  for fila in range(pesos.shape[0]):
    for columna in range(pesos.shape[1]):
      ax[1].text(columna, fila, f"{pesos[fila, columna]:.0f}", ha="center", va="center", color="red" if pesos[fila, columna] > 1.5 else "orange", fontsize=14)
  ax[1].set_title(titulo)
  ax[1].axis("off")

  plt.tight_layout()
  plt.show()

mostrar_pesos(PESOS_DEL_PAPER, "Pesos del paper (0 negro, 4 blanco)")

Ahora la pregunta importante: **¿mejoran los pesos el reconocimiento?** Comparemos la precisión con pesos uniformes contra la del paper en cada una de las $20$ parejas. Cada casillero es una pareja: la fila es la foto de la galería y la columna la foto del probe.

In [ ]:
def mostrar_comparacion_de_pesos(pesos_a, titulo_a, pesos_b, titulo_b):
  precisiones_a = evaluar_pesos(pesos_a)
  precisiones_b = evaluar_pesos(pesos_b)
  nombres_fotos = [f"0{foto + 1}" for foto in FOTOS_EVALUADAS]

  fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
  matrices = [(precisiones_a, f"{titulo_a}: {np.nanmean(precisiones_a) * 100:.1f}%", "viridis", 0, 1),
              (precisiones_b, f"{titulo_b}: {np.nanmean(precisiones_b) * 100:.1f}%", "viridis", 0, 1),
              (precisiones_b - precisiones_a, f"Diferencia ({titulo_b} - {titulo_a}): {(np.nanmean(precisiones_b) - np.nanmean(precisiones_a)) * 100:+.1f} puntos", "RdYlGn", -0.15, 0.15)]
  for eje, (matriz, titulo, mapa_de_colores, minimo, maximo) in zip(ax, matrices):
    eje.imshow(matriz, cmap=mapa_de_colores, vmin=minimo, vmax=maximo)
    for fila in range(len(nombres_fotos)):
      for columna in range(len(nombres_fotos)):
        if fila != columna: eje.text(columna, fila, f"{matriz[fila, columna] * 100:+.0f}" if mapa_de_colores == "RdYlGn" else f"{matriz[fila, columna] * 100:.0f}", ha="center", va="center", color="black" if mapa_de_colores == "RdYlGn" else "white", fontsize=11)
    eje.set_xticks(range(len(nombres_fotos)), nombres_fotos)
    eje.set_yticks(range(len(nombres_fotos)), nombres_fotos)
    eje.set_xlabel("foto del probe")
    eje.set_ylabel("foto de la galeria")
    eje.set_title(titulo)

  plt.tight_layout()
  plt.show()

  return precisiones_a, precisiones_b

precisiones_uniformes, precisiones_del_paper = mostrar_comparacion_de_pesos(PESOS_UNIFORMES, "Uniformes", PESOS_DEL_PAPER, "Paper")

Veamos cómo se ven los aciertos y los errores. Elijan una pareja (la foto de la galería y la del probe) y los pesos. Para cada ejemplo se muestra la foto del **probe** y la persona de la galería que eligió el sistema. En los errores se muestra además la persona **correcta**.

In [ ]:
PESOS_POR_NOMBRE = {'paper': PESOS_DEL_PAPER, 'uniformes': PESOS_UNIFORMES}

def mostrar_ejemplos_de_identificacion(foto_galeria, foto_probe, nombre_pesos, cantidad=4, semilla=0):
  if foto_galeria == foto_probe:
    print("La foto de la galeria y la del probe tienen que ser distintas")
    return

  identificaciones = identificar(CUBOS_ROSTROS[(foto_galeria, foto_probe)], PESOS_POR_NOMBRE[nombre_pesos])
  aciertos = np.flatnonzero(identificaciones == np.arange(200))
  errores = np.flatnonzero(identificaciones != np.arange(200))

  generador = np.random.default_rng(semilla)
  aciertos_elegidos = generador.permutation(aciertos)[:cantidad]
  errores_elegidos = generador.permutation(errores)[:cantidad]

  fig, ax = plt.subplots(5, cantidad, figsize=(2.4 * cantidad, 12.5), squeeze=False)
  for eje in ax.ravel(): eje.axis("off")

  for columna, persona in enumerate(aciertos_elegidos):
    ax[0, columna].imshow(imagenes_feret[persona, foto_probe], cmap="gray", vmin=0, vmax=255)
    ax[0, columna].set_title(f"Probe: persona {persona + 1}", fontsize=10)
    ax[1, columna].imshow(imagenes_feret[identificaciones[persona], foto_galeria], cmap="gray", vmin=0, vmax=255)
    ax[1, columna].set_title(f"Elegida: persona {identificaciones[persona] + 1}", fontsize=10, color="green")

  for columna, persona in enumerate(errores_elegidos):
    ax[2, columna].imshow(imagenes_feret[persona, foto_probe], cmap="gray", vmin=0, vmax=255)
    ax[2, columna].set_title(f"Probe: persona {persona + 1}", fontsize=10)
    ax[3, columna].imshow(imagenes_feret[identificaciones[persona], foto_galeria], cmap="gray", vmin=0, vmax=255)
    ax[3, columna].set_title(f"Elegida: persona {identificaciones[persona] + 1}", fontsize=10, color="red")
    ax[4, columna].imshow(imagenes_feret[persona, foto_galeria], cmap="gray", vmin=0, vmax=255)
    ax[4, columna].set_title(f"Correcta: persona {persona + 1}", fontsize=10)

  fig.suptitle(f"Galeria foto 0{foto_galeria + 1}, probe foto 0{foto_probe + 1}, pesos {nombre_pesos}: {len(aciertos)} aciertos y {len(errores)} errores de 200\nArriba aciertos (probe y elegida), abajo errores (probe, elegida y correcta)")
  plt.tight_layout()
  plt.show()

opciones_fotos = [(f"0{foto + 1}", foto) for foto in FOTOS_EVALUADAS]
interact(
  mostrar_ejemplos_de_identificacion,
  foto_galeria=widgets.Dropdown(options=opciones_fotos, value=0, description='galeria'),
  foto_probe=widgets.Dropdown(options=opciones_fotos, value=6, description='probe'),
  nombre_pesos=widgets.Dropdown(options=list(PESOS_POR_NOMBRE), description='pesos'),
  cantidad=widgets.fixed(4),
  semilla=IntSlider(min=0, max=20, step=1, value=0, description='semilla', continuous_update=False),
);

### Tests (pesos del paper)

In [ ]:
def verificar_pesos_del_paper():
  media_uniformes = np.nanmean(precisiones_uniformes)
  media_del_paper = np.nanmean(precisiones_del_paper)
  parejas_que_mejoran = int((precisiones_del_paper > precisiones_uniformes).sum())
  parejas_que_empeoran = int((precisiones_del_paper < precisiones_uniformes).sum())

  assert media_del_paper > media_uniformes, f"con los pesos del paper se esperaba una precision mayor que con pesos uniformes pero se obtuvo {media_del_paper * 100:.1f}% contra {media_uniformes * 100:.1f}%"
  assert parejas_que_mejoran > parejas_que_empeoran, f"los pesos del paper tienen que mejorar mas parejas de las que empeoran pero mejoran {parejas_que_mejoran} y empeoran {parejas_que_empeoran}"
  print(f"OK [paper] uniformes: {media_uniformes * 100:.1f}%, paper: {media_del_paper * 100:.1f}%, mejora {parejas_que_mejoran} de 20 parejas y empeora {parejas_que_empeoran}")

### Verificacion pesos del paper

In [ ]:
verificar_pesos_del_paper()

Los pesos del paper ganan en la mayoría de las parejas y en las que pierden la diferencia es chica. Las mayores mejoras están en las parejas que usan la foto $07$, la de otra iluminación: hasta $15$ puntos. Con pesos uniformes el pelo, el fondo y las sombras del costado de la cara pesan tanto como los ojos y son justamente lo que más cambia con la luz.

Veamos cuánto de esa mejora viene de los ojos. En este explorador partimos de pesos uniformes y le subimos el peso a las celdas de los ojos (que en el mapa del paper valen $4$) y a las de la boca (que valen $2$). Fijense cómo sube la precisión al darle más peso a los ojos, y cómo deja de subir si se exagera.

In [ ]:
def explorar_pesos(peso_ojos, peso_boca):
  pesos = np.ones((GRILLA_ROSTROS, GRILLA_ROSTROS))
  pesos[1, [1, 2, 4, 5]] = peso_ojos
  pesos[5, [2, 3, 4]] = peso_boca

  precisiones = evaluar_pesos(pesos)
  fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))

  ax[0].imshow(pesos, cmap="gray", vmin=0, vmax=8)
  for fila in range(GRILLA_ROSTROS):
    for columna in range(GRILLA_ROSTROS):
      ax[0].text(columna, fila, f"{pesos[fila, columna]:.0f}", ha="center", va="center", color="red", fontsize=13)
  ax[0].set_title("Pesos")
  ax[0].axis("off")

  barras = ax[1].bar(["uniformes", "paper", "estos pesos"], [np.nanmean(precisiones_uniformes) * 100, np.nanmean(precisiones_del_paper) * 100, np.nanmean(precisiones) * 100], color=["gray", "tab:green", "tab:blue"])
  ax[1].bar_label(barras, fmt="%.1f%%")
  ax[1].set_ylim(40, 80)
  ax[1].set_ylabel("precision media (20 parejas)")
  ax[1].set_title("Precision con estos pesos")

  plt.tight_layout()
  plt.show()

interact(
  explorar_pesos,
  peso_ojos=IntSlider(min=0, max=8, step=1, value=1, description='peso ojos'),
  peso_boca=IntSlider(min=0, max=8, step=1, value=1, description='peso boca'),
);

#### **Robustez a la iluminación**

Vamos a comparar el reconocimiento con LBP contra un método ingenuo: comparar las **imágenes píxel a píxel** (distancia euclídea). Tomamos la foto $06$ como probe y le **cambiamos la iluminación** con una curva gamma ($I' = 255 \cdot (I/255)^\gamma$), que oscurece o aclara la imagen sin cambiar el orden entre los píxeles. La galería es la foto $01$, sin modificar.

In [ ]:
def cambiar_iluminacion(imagen, gamma):
  # Una funcion creciente aplicada a todos los pixeles: cambia los brillos pero no el orden entre ellos
  return 255 * (imagen / 255) ** gamma

def precision_con_pixeles(imagenes_probe):
  galeria = imagenes_feret[:, 0].reshape(200, -1).astype(float)
  probes = imagenes_probe.reshape(200, -1)

  # |a - b|^2 = |a|^2 + |b|^2 - 2 a.b nos ahorra armar un arreglo de 200 x 200 x 6400
  distancias_al_cuadrado = (probes ** 2).sum(axis=1)[:, None] + (galeria ** 2).sum(axis=1)[None, :] - 2 * probes @ galeria.T
  return (np.argmin(distancias_al_cuadrado, axis=1) == np.arange(200)).mean()

def precision_con_lbp(imagenes_probe):
  # skimage avisa que las imagenes float pueden dar empates raros, pero como la transformacion es creciente los empates se conservan
  with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    descriptores_probe = np.array([descriptor_de_rostro(imagen) for imagen in imagenes_probe])

  return calcular_precision(cubo_de_distancias(descriptores_probe, DESCRIPTORES_ROSTROS[0]), PESOS_DEL_PAPER)

def explorar_iluminacion(gamma):
  probes = cambiar_iluminacion(imagenes_feret[:, 5].astype(float), gamma)
  precision_pixeles = precision_con_pixeles(probes)
  precision_lbp = precision_con_lbp(probes)

  fig, ax = plt.subplots(1, 2, figsize=(10, 4))

  ax[0].imshow(probes[0], cmap="gray", vmin=0, vmax=255)
  ax[0].set_title(f"Un probe con gamma = {gamma}")
  ax[0].axis("off")

  barras = ax[1].bar(["pixel a pixel", "LBP"], [precision_pixeles * 100, precision_lbp * 100], color=["gray", "tab:green"])
  ax[1].bar_label(barras, fmt="%.1f%%")
  ax[1].set_ylim(0, 105)
  ax[1].set_ylabel("precision (%)")
  ax[1].set_title("Precision sobre 200 personas")

  plt.tight_layout()
  plt.show()

interact(explorar_iluminacion, gamma=widgets.FloatSlider(min=0.3, max=3, step=0.1, value=1, description='gamma', continuous_update=False));

Y veamos las caras. Para algunas personas elegidas al azar mostramos la foto del probe con la iluminación cambiada y la persona de la galería que eligió cada método, el de píxel a píxel y LBP.

In [ ]:
from scipy.spatial.distance import cdist

def identificar_con_pixeles(imagenes_probe):
  galeria = imagenes_feret[:, 0].reshape(200, -1).astype(float)
  return np.argmin(cdist(imagenes_probe.reshape(200, -1), galeria), axis=1)

def identificar_con_lbp(imagenes_probe):
  with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    descriptores_probe = np.array([descriptor_de_rostro(imagen) for imagen in imagenes_probe])

  return identificar(cubo_de_distancias(descriptores_probe, DESCRIPTORES_ROSTROS[0]), PESOS_DEL_PAPER)

def mostrar_ejemplos_de_iluminacion(gamma, cantidad=4, semilla=0):
  probes = cambiar_iluminacion(imagenes_feret[:, 5].astype(float), gamma)
  identificaciones = {'pixel a pixel': identificar_con_pixeles(probes), 'LBP': identificar_con_lbp(probes)}
  personas = np.random.default_rng(semilla).choice(200, cantidad, replace=False)

  fig, ax = plt.subplots(cantidad, 3, figsize=(8, 2.6 * cantidad), squeeze=False)
  for fila, persona in enumerate(personas):
    ax[fila, 0].imshow(probes[persona], cmap="gray", vmin=0, vmax=255)
    ax[fila, 0].set_title(f"Probe: persona {persona + 1}", fontsize=10)
    for columna, (metodo, elegidas) in enumerate(identificaciones.items(), start=1):
      es_correcta = elegidas[persona] == persona
      ax[fila, columna].imshow(imagenes_feret[elegidas[persona], 0], cmap="gray", vmin=0, vmax=255)
      ax[fila, columna].set_title(f"{metodo}: persona {elegidas[persona] + 1}\n({'correcta' if es_correcta else 'incorrecta'})", fontsize=10, color="green" if es_correcta else "red")
  for eje in ax.ravel(): eje.axis("off")

  fig.suptitle(f"gamma = {gamma}: la persona de la galeria que elige cada metodo")
  plt.tight_layout()
  plt.show()

interact(
  mostrar_ejemplos_de_iluminacion,
  gamma=widgets.FloatSlider(min=0.3, max=3, step=0.1, value=2, description='gamma', continuous_update=False),
  cantidad=widgets.fixed(4),
  semilla=IntSlider(min=0, max=20, step=1, value=0, description='semilla', continuous_update=False),
);

Fijense que el método ingenuo cae apenas se aleja de $\gamma = 1$ mientras que la precisión de LBP **no se mueve**: como la curva gamma es creciente, nunca cambia quién es mayor que quién y por lo tanto los códigos LBP son idénticos. 

Si quieren ver el límite del método, agreguen la $02$ y la $05$ a `FOTOS_EVALUADAS` (acuérdense que la numeración arranca en $0$) y vuelvan a correr desde esa celda: con la cabeza tan girada la precisión cae mucho, porque cada celda deja de mirar la misma parte de la cara.